# Notebook final — Modélisation pré-voyage et post-voyage

Ce notebook reprend les étapes stabilisées du projet IA de planification de séjours :

1. chargement du dataset ;
2. contrôles de cohérence métier ;
3. nettoyage et préparation des données ;
4. feature engineering ;
5. modèle pré-voyage ;
6. modèle post-voyage ;
7. comparaison, choix du modèle et préparation de l'étape suivante.

Les notebooks d'expérimentation restent conservés séparément. Ce notebook sert de version propre pour la suite du projet.


## 1. Cadrage et analyse du besoin

### 1.1 Définition du problème métier

L'agence de voyages souhaite améliorer la personnalisation des séjours et la satisfaction client à partir de l'historique des voyages. Le problème métier concret est double :

- avant le départ, estimer si les informations disponibles lors de la planification suffisent à anticiper la satisfaction client ;
- après le séjour, identifier les facteurs opérationnels associés à une satisfaction faible ou élevée afin d'améliorer les futures propositions.

La cible métier étudiée est `satisfaction_client`. Le projet distingue donc volontairement deux cas d'usage : un objectif pré-voyage, utile pour la planification, et un objectif post-voyage, utile pour l'analyse qualité et l'amélioration continue.

### 1.2 Objectifs IA retenus et règles de fuite de données

| Objectif | Moment d'utilisation | Variables autorisées | Variables exclues |
| --- | --- | --- | --- |
| Pré-voyage | Avant le départ | profil, budget, destination, saison, durée, hébergement, météo prévue, activité | `imprevus`, `reorganisation_necessaire`, `respect_budget`, `retour_client` |
| Post-voyage | Après ou pendant le séjour | variables pré-voyage + événements opérationnels observés | `retour_client` brut et variables NLP dérivées |

Pour le modèle post-voyage final, la satisfaction est regroupée en 3 classes :

- `0` : insatisfait, notes 1 et 2 ;
- `1` : neutre, note 3 ;
- `2` : satisfait, notes 4 et 5.

Cette séparation évite la fuite de données : les variables connues uniquement pendant ou après le séjour ne doivent pas être utilisées pour prédire la satisfaction avant le départ.

### 1.3 Indicateurs de succès du projet

| KPI | Rôle dans le projet | Seuil ou interprétation attendue |
| --- | --- | --- |
| `macro_f1` | Mesurer la performance globale en tenant compte des classes minoritaires | Indicateur principal pour comparer les modèles |
| `balanced_accuracy` | Vérifier que le modèle ne favorise pas uniquement la classe majoritaire | Doit être supérieur au modèle naïf `Dummy` |
| Matrice de confusion | Identifier les classes les plus confondues | La classe `neutre` est surveillée spécifiquement |
| Stabilité en validation croisée | Vérifier que le score ne dépend pas d'un seul découpage train/test | Écart-type faible recherché |
| Cohérence métier des variables importantes | Vérifier que le modèle s'appuie sur des signaux interprétables | Les variables post-voyage doivent ressortir dans l'objectif post-voyage |
| Absence de fuite de données | Garantir que le modèle respecte le moment d'utilisation prévu | Obligatoire pour comparer pré-voyage et post-voyage |

### 1.4 Contraintes du projet

| Contrainte | Impact sur le projet |
| --- | --- |
| Données synthétiques | Les résultats servent de preuve de méthode, pas de validation opérationnelle sur clients réels. |
| Volume limité, environ 1500 lignes | Les modèles complexes risquent de surapprendre et les performances restent sensibles au split train/test. |
| Ressources machine limitées | Les traitements trop lourds, notamment NLP avancé ou optimisation massive, doivent être évités ou isolés. |
| Délai de projet de certification | Priorité à une démarche claire, reproductible et documentée plutôt qu'à une optimisation excessive. |
| Contraintes RGPD | Le projet actuel utilise des données synthétiques ; toute donnée réelle nécessiterait un cadrage RGPD renforcé. |
| Industrialisation reportée | Le notebook final doit rester propre et complet avant de figer un pipeline de production. |

### 1.5 L'IA est-elle nécessaire ?

Une solution simple suffit pour certaines parties du projet : règles métier, tableaux de bord, contrôles de cohérence et statistiques descriptives permettent déjà d'identifier des anomalies et des facteurs de risque.

L'IA devient pertinente uniquement si l'objectif est de généraliser ces observations à de nouveaux séjours et de produire une estimation automatisée de la satisfaction. Les expériences du notebook montrent cependant que :

- le cas pré-voyage contient peu de signal prédictif avec les variables disponibles ;
- le cas post-voyage est plus cohérent pour l'analyse qualité, car il utilise les événements réellement observés pendant le séjour ;
- le modèle doit rester une aide à la décision, et non un système de décision automatique.


## 2. Documentation metier et datasheet

objectif dataset, datasheet, variables principales et principes de preparation des donnees.


### 2.1 Datasheet structurée du jeu de données

Cette sous-section formalise la datasheet du fichier `data/Examen_travel_planning_dataset.csv` selon les dimensions attendues pour documenter un jeu de données utilisé dans un projet IA.

#### Motivation et contexte de collecte

Le dataset est utilisé dans le cadre d'un projet de planification de voyages. L'objectif métier est d'analyser des séjours passés afin de comprendre les facteurs associés à la satisfaction client et d'évaluer la possibilité de construire un modèle de prédiction. Les données sont synthétiques et anonymisées ; elles ne proviennent donc pas d'une collecte client réelle, mais simulent des situations plausibles de voyages avec budgets, destinations, contraintes, imprévus et retours clients.

#### Composition et statistiques descriptives

Le fichier contient environ 1500 lignes et 15 colonnes. Chaque ligne représente un séjour. Les variables couvrent le profil client, le budget, la destination, la saison, la durée, l'hébergement, le prix du vol, la météo prévue, l'activité principale, les imprévus, la réorganisation, le respect du budget, la satisfaction client et le retour textuel. Les statistiques descriptives sont vérifiées dans le notebook : valeurs manquantes, distribution de la cible, types de variables, valeurs uniques, doublons, incohérences métier et valeurs atypiques.

#### Usages recommandés

Le dataset est adapté à l'exploration de données, à l'analyse de cohérence métier, à la comparaison entre un objectif pré-voyage et un objectif post-voyage, ainsi qu'à la construction d'un prototype de modèle IA. Pour le modèle pré-voyage, seules les variables disponibles avant le séjour doivent être utilisées. Pour le modèle post-voyage, les variables d'événements observés comme `imprevus`, `reorganisation_necessaire` et `respect_budget` peuvent être utilisées si l'objectif est l'analyse ou l'évaluation après séjour.

#### Limitations

Le dataset reste synthétique, de taille modérée et ne reflète pas nécessairement toute la complexité d'une activité réelle d'agence de voyages. Certaines variables pré-voyage contiennent peu de signal pour expliquer la satisfaction client. Plusieurs incohérences doivent être contrôlées avant modélisation. Les variables post-voyage ne doivent pas être utilisées dans un modèle censé prédire la satisfaction avant le départ, afin d'éviter une fuite de données.

#### Distribution

Le fichier est stocké localement dans le projet au chemin `data/Examen_travel_planning_dataset.csv`. Comme il s'agit d'un dataset synthétique et anonymisé fourni pour le projet, il peut être versionné dans GitHub dans le cadre de cette certification. Si des données réelles étaient ajoutées ultérieurement, une vérification RGPD, une anonymisation et une politique d'accès seraient nécessaires avant toute diffusion.

#### Maintenance

Toute modification du dataset doit être documentée : ajout de colonnes, correction d'incohérences, enrichissement externe, suppression de lignes ou changement de cible. Les traitements appliqués doivent rester reproductibles dans le notebook. En cas d'évolution du jeu de données, il faudra mettre à jour les statistiques descriptives, les contrôles de cohérence, la liste des variables autorisées et les résultats de modélisation.


### 2.2 Registre de traitement (RGPD)

Dans ce projet, le traitement porte uniquement sur le fichier `data/Examen_travel_planning_dataset.csv`. Ce fichier est un dataset synthétique et anonymisé fourni pour la certification. Il ne contient pas de nom, prénom, email, téléphone, adresse, numéro de passeport ou identifiant client réel. Le registre ci-dessous documente donc le traitement réalisé dans ce projet, et non un traitement opérationnel de données clients réelles.

#### Finalités du traitement

| Finalité | Description |
| --- | --- |
| Documentation du dataset | Décrire le jeu de données utilisé pour répondre au besoin métier de planification de voyages. |
| Analyse de cohérence métier | Vérifier la qualité du fichier : valeurs manquantes, incohérences, doublons, valeurs atypiques. |
| Modélisation pré-voyage | Tester la capacité des variables disponibles avant le séjour à expliquer la satisfaction client. |
| Modélisation post-voyage | Tester l'apport des variables observées pendant ou après le séjour : `imprevus`, `respect_budget`, `reorganisation_necessaire`. |

#### Catégories de données collectées

| Catégorie | Colonnes concernées | Statut dans ce projet |
| --- | --- | --- |
| Identifiant technique | `trip_id` | Identifiant synthétique de ligne, non rattaché à une personne réelle. |
| Profil voyageur simulé | `client_type` | Catégorie générique : famille, couple, solo, business, senior. |
| Caractéristiques du séjour | `destination`, `saison`, `duree_jours`, `type_hebergement`, `activite_principale`, `meteo_prevue` | Variables descriptives du voyage fictif. |
| Données budgétaires simulées | `budget_total`, `prix_vol`, `respect_budget` | Montants fictifs utilisés pour l'analyse et la modélisation. |
| Événements post-voyage simulés | `imprevus`, `reorganisation_necessaire` | Variables opérationnelles fictives connues après ou pendant le séjour. |
| Satisfaction et avis fictifs | `satisfaction_client`, `retour_client` | Score et commentaire synthétiques, sans auteur identifiable. |

Aucune donnée personnelle directement identifiable n'est traitée dans le périmètre actuel du notebook.

#### Durée de conservation

| Élément conservé | Durée retenue pour ce projet |
| --- | --- |
| Dataset brut synthétique | Conservé dans `data/` pendant la durée du projet. |
| Notebooks d'analyse | Conservés dans `notebooks/` pour assurer la traçabilité des choix et des résultats. |
| Documentation projet | Conservée dans `docs/` et dans le notebook final pour justifier la démarche. |

Comme le dataset est synthétique, il n'y a pas de durée de conservation liée à des personnes concernées identifiables dans ce projet.

#### Mesures de sécurité

- Le projet ne versionne que le dataset synthétique fourni pour la certification.
- Aucune donnée client réelle ne doit être ajoutée dans `data/` ou poussée sur GitHub dans cette version du projet.
- Les transformations appliquées au dataset sont documentées dans le notebook afin de conserver une traçabilité complète.

#### Droits des personnes concernées

Dans le périmètre actuel, aucune personne physique identifiable n'est présente dans le dataset. Les droits RGPD individuels ne sont donc pas directement applicables au fichier synthétique utilisé dans ce projet.

Si le périmètre change et que des données clients réelles sont ajoutées, ce registre devra être remplacé par un registre RGPD opérationnel validé avant toute modélisation.


### 2.3 Analyse des risques ethiques et societaux de la solution IA

Cette section identifie les risques ethiques et societaux lies a l'exploitation de la solution IA du projet de planification de voyages. Elle complete la datasheet, le registre RGPD et la Model Card.

#### Referentiels éthiques et réglementaires pris en compte

| Referentiel | Apport pour le projet | Application dans ce notebook |
| --- | --- | --- |
| Lignes directrices europeennes pour une IA digne de confiance | Cadre europeen fondé sur l'action humaine, la robustesse, la vie privee, la transparence, l'équité, le bien-etre societal et la responsabilite | Vérification de la fuite de données, interpretation des limites, conservation d'un role humain dans la decision |
| AI Act europeen, reglement UE 2024/1689 | Approche par les risques et exigence de systemes IA surs, transparents et respectueux des droits fondamentaux | Le projet reste un prototype de certification ; tout deploiement reel necessiterait une qualification du niveau de risque |
| Recommandations IA et RGPD de la CNIL | Points de controle sur les donnees, les droits des personnes, l'information et la conformite RGPD | Le dataset est synthetique ; le notebook documente neanmoins les finalites, categories de donnees et mesures de securite |
| Travaux CNIL / Defenseur des droits sur algorithmes et discriminations | Rappel que les algorithmes peuvent produire ou renforcer des biais et exclusions | Analyse des biais lies au budget, au profil client, a la destination et aux classes de satisfaction |
| Rapport CNIL "Garder la main" | Importance du controle humain et de la comprehension des decisions algorithmiques | Le modele est presente comme une aide a l'analyse, pas comme une decision automatique |

Sources officielles utilisees : [Commission europeenne - Ethics Guidelines for Trustworthy AI](https://digital-strategy.ec.europa.eu/en/library/ethics-guidelines-trustworthy-ai), [EUR-Lex - Regulation EU 2024/1689 Artificial Intelligence Act](https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng), [CNIL - fiches pratiques IA](https://www.cnil.fr/fr/les-fiches-pratiques-ia), [CNIL - recommandations IA et RGPD](https://www.cnil.fr/fr/developpement-des-systemes-dia-les-recommandations-de-la-cnil-pour-respecter-le-rgpd), [CNIL / Defenseur des droits - algorithmes et discriminations](https://www.cnil.fr/fr/algorithmes-et-discriminations-le-defenseur-des-droits-avec-la-cnil-appelle-une-mobilisation).

#### Impacts ethiques et societaux identifies

| Risque | Consequence possible | Niveau dans ce projet | Mesure de prise en compte |
| --- | --- | --- | --- |
| Utilisation d'un modele peu performant | Recommandations ou analyses erronees si les scores sont interpretes comme fiables individuellement | Moyen | Comparaison avec baseline, validation croisee, matrice de confusion, documentation des limites |
| Confusion entre pre-voyage et post-voyage | Utiliser des informations connues apres le sejour pour prendre une decision avant depart | Eleve si mal utilise | Separation explicite des objectifs et exclusion des variables post-voyage du modele pre-voyage |
| Automatisation excessive | Decisions commerciales prises sans analyse humaine | Moyen | Positionnement du modele comme aide a la decision uniquement |
| Segmentation injuste des clients | Risque de favoriser certains profils ou budgets au detriment d'autres voyageurs | Moyen | Identification des variables sensibles indirectes : `budget_total`, `client_type`, `destination`, `type_hebergement` |
| Manque de representativite du dataset | Resultats non generalisables a une agence reelle | Eleve | Mention explicite du caractere synthetique, anonymise et limite du dataset |
| Mauvaise interpretation des commentaires clients | Le texte libre peut refleter directement la satisfaction ou contenir des biais de langage | Moyen | `retour_client` exclu du modele principal pour eviter fuite de donnees et surinterpretation |
| Impact environnemental des experimentations | Consommation inutile de ressources si multiplication de modeles lourds | Faible a moyen | Modeles limites, `n_jobs=1`, experimentation NLP lourde non retenue dans le modele principal |

#### Biais potentiels ou existants

| Biais identifie | Origine possible | Impact metier | Traitement dans le projet |
| --- | --- | --- | --- |
| Biais socio-economique | `budget_total`, `prix_vol`, `type_hebergement`, `destination_luxe` peuvent refleter un niveau de depense | Le modele pourrait associer satisfaction et niveau de budget plutot que qualite reelle du service | Analyse des importances de variables et prudence dans l'interpretation |
| Biais de profil client | `client_type` distingue famille, couple, solo, business, senior | Risque de recommandations stereotypees selon le profil | Variable conservee mais a surveiller par sous-groupes en cas de deploiement |
| Biais de destination | Certaines destinations peuvent etre associees a luxe, distance ou meteo | Risque de favoriser ou penaliser certaines destinations | Enrichissement limite et documente ; certaines variables supprimees de la modelisation |
| Biais de classe cible | Les classes de satisfaction ne sont pas parfaitement equilibrees | Le modele peut mieux predire les classes majoritaires que les classes minoritaires | Controle de l'equilibre des classes, `macro_f1`, `balanced_accuracy`, matrice de confusion |
| Biais de donnees synthetiques | Le dataset ne provient pas d'observations reelles | Les relations apprises peuvent etre artificielles | Limitation clairement documentee dans la datasheet et la Model Card |
| Biais de variables post-voyage | `imprevus`, `respect_budget`, `reorganisation_necessaire` sont connues apres ou pendant le sejour | Risque de fuite si utilisees pour un usage pre-voyage | Variables exclues du modele pre-voyage et autorisees uniquement dans l'objectif post-voyage |

#### Dilemmes ethiques identifies

| Dilemme | Description | Position retenue |
| --- | --- | --- |
| Personnalisation vs discrimination | Personnaliser un voyage peut ameliorer l'experience, mais aussi enfermer certains profils dans des recommandations stereotypees | Utiliser le modele comme aide, pas comme regle automatique de proposition |
| Optimisation commerciale vs interet client | Le modele pourrait etre utilise pour privilegier les sejours les plus rentables plutot que les plus adaptes | Garder la satisfaction client comme cible centrale et documenter les variables budgetaires |
| Performance vs explicabilite | Des modeles plus complexes peuvent parfois mieux performer mais etre moins interpretables | Privilegier des modeles comparables, une matrice de confusion, les importances de variables et une Model Card |
| Donnees post-voyage vs prediction avant depart | Les variables post-voyage ameliorent les scores mais ne sont pas disponibles avant le depart | Separer strictement les objectifs pre-voyage et post-voyage |
| Automatisation vs controle humain | Automatiser l'analyse peut accelerer le traitement mais reduire la vigilance metier | Maintenir une validation humaine, surtout pour les decisions concernant un client ou une offre |

#### Acteurs a informer et verifications attendues

| Acteur concerne | Informations a porter a sa connaissance | Verification attendue | Statut dans ce projet |
| --- | --- | --- | --- |
| Commanditaire / agence de voyages | Performances modestes, limites du dataset, usage recommande du modele | Valider que le modele sert d'aide a l'analyse et non de decision automatique | A presenter avec la synthese finale |
| Referent metier | Coherence des variables, regles de nettoyage, interpretation des incoherences | Confirmer que les regles metier sont acceptables | Partiellement traite dans l'analyse metier du notebook |
| Juriste / DPO | Absence de donnees personnelles reelles, conditions en cas de donnees reelles | Verifier la conformite RGPD avant tout passage a des donnees clients reelles | Documente dans le registre RGPD ; validation externe necessaire si deploiement |
| Equipe data / IA | Fuite de donnees, biais, metriques, sous-groupes, reproductibilite | Revoir les features, metriques, seuils et limites techniques | Documente dans le notebook, la Model Card et MLflow |
| Evaluateur certification | Demarche, sources, limites, arbitrages | Verifier que les risques sont identifies et pris en compte | Couvert par cette section et les sections precedentes |

#### Mesures concretes appliquees dans le notebook

- Les objectifs pre-voyage et post-voyage sont separes pour eviter la fuite de donnees.
- Les variables `imprevus`, `respect_budget`, `reorganisation_necessaire` sont exclues du modele pre-voyage.
- Le texte libre `retour_client` est exclu du modele principal pour eviter une fuite directe du ressenti client.
- Les performances sont comparees a un modele naif et evaluees avec `macro_f1`, `balanced_accuracy`, validation croisee et matrice de confusion.
- L'equilibre des classes est controle apres nettoyage, apres feature engineering et sur le modele post-voyage.
- Les limites du dataset synthetique sont documentees dans la datasheet, le registre RGPD et la Model Card.
- La tracabilite experimentale est prevue avec MLflow.

#### Conclusion ethique pour ce projet

Dans son perimetre actuel, la solution IA reste un prototype de certification utilisant un dataset synthetique et anonymise. Le risque legal immediat est limite, mais le risque methodologique et ethique principal concerne la mauvaise interpretation du modele : il ne doit pas etre utilise comme outil de decision automatique ni comme preuve de performance operationnelle. Avant tout deploiement reel, une validation metier, juridique et RGPD serait obligatoire, avec des tests de biais sur donnees reelles et une gouvernance claire des usages.


1. Objectif de l'etape 
L'objectif de cette etape est d'identifier et de justifier un jeu de donnees capable de repondre aux besoins metiers et aux cas d'usage du projet IA. 

Le jeu de donnees etudie est : 

data/Examen_travel_planning_dataset.csv 
 
Il concerne la planification de voyages et contient des informations sur les clients, les destinations, les budgets, les vols, la meteo, les activites, les imprevus et les retours clients. 

2. Datasheet du jeu de donnees 

Cette datasheet synthetise les informations essentielles sur le dataset. 

#### Identification

| Element | Description |
| --- | --- |
| Nom du fichier | `Examen_travel_planning_dataset.csv` |
| Emplacement | `data/Examen_travel_planning_dataset.csv` |
| Format | CSV |
| Domaine metier | Planification de voyages haut de gamme |
| Volume | 1500 lignes, 15 colonnes |
| Acces | Disponible localement dans le projet |
| Nature des donnees | Donnees synthetiques et anonymisees |
| Usage principal | Analyse et prediction liees a la personnalisation des sejours |


3.Description generale 

Le dataset represente des voyages planifies pour differents profils de clients. Chaque ligne correspond a un voyage et contient des informations sur le budget, la destination, la saison, la duree, l'hebergement, le vol, la meteo prevue, les activites, les imprevus, la satisfaction et le respect du budget. 



#### Variables principales

| Colonne | Type metier | Utilite pour le projet |
| --- | --- | --- |
| `trip_id` | Identifiant | Suivi technique, a exclure du modele |
| `client_type` | Categorie client | Adapter les propositions au profil du voyageur |
| `budget_total` | Numerique | Tenir compte de la contrainte budgetaire globale |
| `destination` | Categorie | Comparer les destinations possibles |
| `saison` | Categorie | Tenir compte de la periode du voyage |
| `duree_jours` | Numerique | Evaluer l'impact de la duree du sejour |
| `type_hebergement` | Categorie | Evaluer l'impact du logement sur l'experience |
| `prix_vol` | Numerique | Mesurer l'impact du prix du billet d'avion sur le budget total |
| `meteo_prevue` | Categorie | Anticiper les risques lies a la meteo |
| `activite_principale` | Categorie | Relier le sejour aux centres d'interet du client |
| `satisfaction_client` | Numerique | Cible principale recommandee pour evaluer la qualite du sejour |
| `imprevus` | Categorie | Cible ou variable d'analyse pour comprendre les incidents |
| `reorganisation_necessaire` | Binaire | Indique si un voyage a dû être réorganisé ou pas, Peut servir de cible à prédire  |
| `respect_budget` | Binaire | Indiquer si le budget prévu a été respecté ; cette colonne est une cible principale provisoire |
| `retour_client` | Texte | Exploitable plus tard pour une analyse NLP |


4.Cibles IA possibles 


| Cible | Priorite | Type de probleme | Interet metier |
| --- | --- | --- | --- |
| `satisfaction_client` | Principale recommandee | Regression ou classification ordinale | Predire la qualite attendue d'un sejour et guider la personnalisation |
| `reorganisation_necessaire` | Secondaire | Classification binaire | Anticiper les sejours qui risquent d'etre reorganises |
| `respect_budget` | Secondaire | Classification binaire | Controler le risque de non-respect du budget |
| `imprevus` | Secondaire ou future | Classification multiclasses | Anticiper le type d'incident possible |
| `retour_client` | Future | Analyse de texte | Exploiter les avis clients pour ameliorer les propositions |


5.Qualite et points de vigilance 

| Point controle | Observation |
| --- | --- |
| Fichier accessible | Oui |
| Format lisible | Oui, csv |
| Valeurs manquantes  | Presentes sur plusieurs colonnes |
| Variables categorielles | Plusieurs colonnes a encoder |
| Variables numeriques | Budget, duree, prix du vol, satisfaction |
| Confidentialite | Donnees synthetiques et anonymisees, utilisables dans le cadre de la certification |

6.Usage prevu 

Le dataset est prevu pour : 
-Analyser les facteurs qui influencent la satisfaction client ; 
-Entrainer un premier modele de prediction de satisfaction ; 


7. Besoins metiers identifies 

L'agence de voyages souhaite proposer une planification haut de gamme plus personnalisee, plus fiable et plus reactive face aux imprevus. 

Les besoins metiers identifies sont : 
-recommander des destinations adaptees au profil du voyageur ; 
-tenir compte du budget, des centres d'interet, de la saison et de la duree ; 
-planifier les elements logistiques : vol, hebergement et organisation du sejour ; 
-anticiper les imprevus comme la meteo, les retards, les annulations ou les problemes de bagages ; 
-identifier les voyages qui risquent de necessiter une reorganisation ; 
-ameliorer la satisfaction client ; 
-exploiter les retours clients pour ameliorer les futures propositions ; 

8. Cas d'usage decrits 

Cas d'usage 1 - Recommandation personnalisee de sejour 
Le modele predit la satisfaction attendue pour differentes options de sejour. 
Variable cible principale : 
satisfaction_client 
Exemple d'utilisation : 
Pour un client de type couple, avec un budget donné, une saison, une duree et des activites preferees, le modele peut comparer plusieurs destinations ou types d'hebergement et estimer la satisfaction attendue. 

Cas d'usage 2 - Anticipation des reorganisation et imprevus 
Le modele peut aider a estimer si un sejour risque de necessiter une reorganisation. 
Variable cible possible : 
reorganisation_necessaire 
Autre cible possible : 
imprevus 
Exemple d'utilisation : 
Avant le depart, l'agence identifie les voyages sensibles selon la destination, la saison, la meteo prevue, le type d'hebergement ou le type d'activite. 

Cas d'usage 3 - Controle du respect du budget 
Le modele peut apprendre a predire si un voyage respectera le budget prevu. 
Variable cible possible : 
respect_budget 

Exemple d'utilisation : 
Un conseiller renseigne le profil du client, la destination, la saison, la duree, le type d'hebergement et le prix du vol. Le modele indique si le sejour presente un risque de non-respect du budget. 
 
Cas d'usage 4 - Amelioration continue avec les retours clients 
Le texte libre des retours clients peut etre exploite dans une phase future. 
Variable exploitable : 
retour_client 

Exemple d'utilisation : 
Les commentaires clients peuvent etre analyses pour extraire des themes recurrents : satisfaction, deception, probleme logistique, qualite de l'hebergement ou meteo. 


5. Donnees disponibles dans le fichier
 

6. Choix de la cible principale 

Compte tenu du contexte, la cible la plus pertinente est satisfaction_client. 

Justification metier 

Le projet demande une solution centree sur la satisfaction client. L'objectif n'est pas seulement de respecter un budget ou d'eviter une reorganisation, mais de proposer un sejour personnalise et de qualite. 

satisfaction_client mesure directement le resultat metier recherche : l'experience finale du client. 

Justification IA 

Cette cible permet de construire un modele capable d'estimer la satisfaction attendue pour une proposition de sejour. Le conseiller peut ensuite comparer plusieurs options et choisir celle qui maximise la satisfaction tout en respectant les contraintes.

7. Donnees pertinentes pour alimenter le modele IA 

Pour un modele de prediction de satisfaction, les donnees a priori pertinentes sont les informations disponibles avant ou pendant la planification du sejour : (9 colonnes) 

client_type ; 
budget_total ; 
destination ; 
saison ; 
duree_jours ; 
type_hebergement ; 
prix_vol ; 
meteo_prevue ; 
activite_principale; 
Variable cible : satisfaction_client 

8. Donnees necessaires pour le modèle IA  

| Donnee necessaire | Pourquoi elle est utile |
| --- | --- |
| client_type | Adapter la recommandation au profil du voyageur |
| budget_total | Respecter la contrainte budgetaire |
| destination | Comparer les options de sejour |
| saison | Tenir compte de la periode du voyage |
| duree_jours | Adapter la proposition a la duree disponible |
| type_hebergement | Evaluer l'impact du logement sur l'experience |
| prix_vol | Integrer le cout du transport |
| activite_principale | Representer le centre d'interet principal |
| satisfaction_client | Variable cible pour entrainer le modele |


9. Pertinence et representativite du dataset 

Le dataset est pertinent pour le projet car il couvre plusieurs dimensions metier importantes : 

profils clients : business, solo, senior, couple, famille ; 
destinations variees : New York, Rome, Lisbonne, Bali, Paris, Dubai, Tokyo, Sydney ; 
saisons : automne, ete, printemps, hiver ; 
types d'hebergement : villa, appartement, hotel, resort ; 
meteo prevue : ensoleille, pluie, variable, nuageux ; 
activites : culture, plage, business, gastronomie, randonnee ; 
imprevus : meteo, retard de vol, bagages, annulation, aucun. 

Repartition observee de quelques variables : 
| Variable | Repartition observee |
| --- | --- |
| client_type  | 5 profils clients differents |
| destination | 8 destinations differentes |
| saison | 4 saisons representees  |
| respect_budget | 1041 valeurs 1, 459 valeurs 0 |
| reorganisation_necessaire | 830 valeurs 1, 670 valeurs 0 |

10. Verification de coherence initiale 
Une premiere verification du fichier a ete realisee. 
Valeurs numeriques observees :

| Colonne | Minimum | Maximum | Moyenne |
| --- | --- | --- | --- |
| budget_total | 380.0 | 41000.0 | 7194.04 |
| duree_jours | 2 | 42 | 10.45 |
| prix_vol | 35.0 | 5200.0 | 1125.81 |
| satisfaction_client | 0.0 | 7.0 | 2.78 |

Distribution observee de satisfaction_client : 

| Valeur | Nombre de lignes |
| --- | ---: |
| Valeur manquante | 25 |
| `0.0` | 1 |
| `1.0` | 258 |
| `2.0` | 427 |
| `3.0` | 353 |
| `4.0` | 258 |
| `5.0` | 174 |
| `6.0` | 3 |
| `7.0` | 1 |

Valeurs manquantes observees :

| Colonne | Valeurs manquantes |
| --- | ---: |
| `budget_total` | 40 |
| `type_hebergement` | 36 |
| `prix_vol` | 53 |
| `meteo_prevue` | 39 |
| `activite_principale` | 48 |
| `satisfaction_client` | 25 |
| `imprevus` | 53 |
| `retour_client` | 25 |

#### Point de coherence sur la satisfaction

Le contexte du projet indique que `satisfaction_client` est un score de `1 a 5`.
Le fichier observe contient cependant quelques valeurs hors de cette echelle :
`0.0`, `6.0` et `7.0`.

Ce point doit etre documente et traite avant l'entrainement :

- soit en corrigeant ou filtrant les valeurs hors echelle ;
- soit en justifiant une echelle differente si elle est confirmee par le metier ;
- soit en regroupant les scores dans des classes de satisfaction.


#### Ajouter les différentes incohérences 
L’incohérence prix_vol > budget_total

Le budget_total représente le budget global du séjour et nous admettons que prix_vol représente seulement le coût du vol aller-retour.
Donc si prix_vol > budget_total, le séjour est incohérent : le transport dépasse déjà tout le budget disponible.
Impact Sur Le Modèle : Cette incohérence peut fausser :
-les relations entre budget et satisfaction ;
-les variables dérivées comme ratio_vol_budget ;
-l’apprentissage du modèle si ces lignes restent dans les données.


L'incohérence prix_vol > budget_total avec respect_budget = 1
prix_vol > budget_total signifie que le vol dépasse déjà le budget total.
respect_budget = 1 signifie que le budget aurait été respecté.
Les deux informations se contredisent.
...

...

...


12. Limites identifiees 

Malgre sa pertinence, le dataset presente plusieurs limites : 
-certaines colonnes contiennent des valeurs manquantes ; 
-le volume reste modéré avec 1500 lignes ; 
-le fichier est un dataset synthetique de projet ; 
-Plusieurs incohérences ont été identifiées et doivent être prises en compte avant la modélisation.
-la colonne retour_client est textuelle et demandera un traitement specifique ; 

13. Solutions alternatives

Alternative pour les données synthétiques 
Les données sont synthétiques il y a possibilité d’enrichir le dataset en cas d’insuffisance de données. 


### Préparation des données
### Traitement des incohérences 

#### Uncité 

#### Validité 
respect du format 

#### Satisfaction_client
Pour satisfaction_client, comme c’est la cible du modèle.
La cible doit être entre 1 et 5.
le traitement le plus pertinent est de :
Supprimer les lignes où satisfaction_client est manquante.
Supprimer les lignes où satisfaction_client n’est pas entre 1 et 5.

#### Prix_vol > budget_total
Comme il s’agit d’une incohérence forte sur deux variables d’entrée, le plus propre est de supprimer ces lignes.
Pourquoi supprimer plutôt que corriger ?
On n’a pas la vraie valeur correcte.
Remplacer prix_vol ou budget_total inventerait une information.
Le nombre de cas reste est de 52 lignes. 

### Prix_vol > budget_total avec respect_budget = 1
Nombre de cas : 5
Cette incohérence est incluse dans le contrôle plus général prix_vol > budget_total. Le traitement précédent supprime donc automatiquement ces cas, sans nécessiter une règle de nettoyage supplémentaire.

### Autres incohérences : 
Les autres incohérences concernent les variables post-séjour : imprevus, reorganisation_necessaire, respect_budget, retour_client
Ces variables sont connues après le voyage.
Or notre modèle cherche à prédire satisfaction_client avant ou au moment de la planification du séjour.
Nous gardons ces colonnes dans le dataset pour l’analyse métier par ailleurs nous les exclurons du modèle pour éviter la fuite des données.


## 3. Imports et configuration


In [1]:
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
import optuna

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MLFLOW_TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("travel_planning_satisfaction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 40)

RANDOM_STATE = 42
TARGET_COLUMN = "satisfaction_client"


c:\Users\khadi\Downloads\Examen_IA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow tracking URI: sqlite:///c:/Users/khadi/Downloads/Examen_IA/mlflow.db


## 4. Chargement du dataset


In [2]:
data_path = Path("..") / "data" / "Examen_travel_planning_dataset.csv"
if not data_path.exists():
    data_path = Path("data") / "Examen_travel_planning_dataset.csv"

df_raw = pd.read_csv(data_path)

print(f"Fichier chargé : {data_path}")
print(f"Dimensions : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes")
display(df_raw.head())

Fichier chargé : ..\data\Examen_travel_planning_dataset.csv
Dimensions : 1500 lignes x 15 colonnes


,trip_id,client_type,budget_total,destination,saison,duree_jours,type_hebergement,prix_vol,meteo_prevue,activite_principale,satisfaction_client,imprevus,reorganisation_necessaire,respect_budget,retour_client
0,1,solo,5548.90,Dubaï,automne,3,hôtel,1320.77,ensoleillé,plage,3.0,annulation,1,0,Séjour mitigé.
1,2,business,3288.27,Rome,printemps,12,hôtel,1524.09,variable,randonnée,4.0,aucun,0,1,Excellent séjour.
2,3,senior,13347.18,Bali,printemps,16,villa,881.94,ensoleillé,plage,5.0,météo,1,1,Excellent séjour.
3,4,couple,7049.42,Sydney,automne,17,appartement,844.39,nuageux,randonnée,3.0,météo,1,1,Séjour mitigé.
4,5,solo,7612.92,Paris,hiver,7,hôtel,1081.24,pluie,gastronomie,2.0,retard_vol,1,1,Séjour mitigé.


## 5. Analyse métier detaillée

Cette section reprend l'analyse metier de coherence du dataset brut : controle de la cible, budget, client business, meteo, imprevus, risques de fuite de donnees et selection des incoherences a traiter avant modelisation.


### Analyse métier de cohérence du dataset brut

Ce notebook documente les contrôles de cohérence métier du fichier `Examen_travel_planning_dataset.csv`.

Objectif : identifier les cas cohérents, suspects ou incohérents avant la préparation des données et la modélisation IA.


### 1. Méthode d'analyse

Les contrôles sont réalisés sur le dataset brut, sans correction préalable.

Pour chaque observation, le notebook contient :

1. l'explication métier du contrôle ;
2. le code de détection ;
3. l'affichage des lignes concernées.

Les cas détectés ne sont pas tous des erreurs. Certains sont des points de vigilance à valider avec le métier.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

In [4]:
# Chargement robuste du fichier CSV depuis le dossier data
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "Examen_travel_planning_dataset.csv"
df_brut = pd.read_csv(DATA_PATH)

print(f"Dataset chargé : {DATA_PATH}")
print(f"Dimensions : {df_brut.shape[0]} lignes x {df_brut.shape[1]} colonnes")
display(df_brut.head())

Dataset chargé : c:\Users\khadi\Downloads\Examen_IA\data\Examen_travel_planning_dataset.csv
Dimensions : 1500 lignes x 15 colonnes


,trip_id,client_type,budget_total,destination,saison,duree_jours,type_hebergement,prix_vol,meteo_prevue,activite_principale,satisfaction_client,imprevus,reorganisation_necessaire,respect_budget,retour_client
0,1,solo,5548.90,Dubaï,automne,3,hôtel,1320.77,ensoleillé,plage,3.0,annulation,1,0,Séjour mitigé.
1,2,business,3288.27,Rome,printemps,12,hôtel,1524.09,variable,randonnée,4.0,aucun,0,1,Excellent séjour.
2,3,senior,13347.18,Bali,printemps,16,villa,881.94,ensoleillé,plage,5.0,météo,1,1,Excellent séjour.
3,4,couple,7049.42,Sydney,automne,17,appartement,844.39,nuageux,randonnée,3.0,météo,1,1,Séjour mitigé.
4,5,solo,7612.92,Paris,hiver,7,hôtel,1081.24,pluie,gastronomie,2.0,retard_vol,1,1,Séjour mitigé.


In [5]:
# Fonctions utilitaires utilisées dans tout le notebook

def normaliser_texte(serie):
    """Standardise les textes pour faciliter les comparaisons métier."""
    return serie.fillna("").astype(str).str.strip().str.lower()


def afficher_cas(df_cas, colonnes=None, n=10):
    """Affiche le nombre de cas détectés puis un échantillon lisible."""
    print(f"Nombre de cas détectés : {len(df_cas)}")
    if len(df_cas) == 0:
        print("Aucun cas à afficher.")
        return

    if colonnes is not None:
        display(df_cas[colonnes].head(n))
    else:
        display(df_cas.head(n))


colonnes_principales = [
    "trip_id", "client_type", "budget_total", "destination", "saison",
    "duree_jours", "type_hebergement", "prix_vol", "meteo_prevue",
    "activite_principale", "satisfaction_client", "imprevus",
    "reorganisation_necessaire", "respect_budget"
]

### 2. Vue d'ensemble du dataset brut

Avant de chercher les incohérences métier, on vérifie la structure générale : volume, colonnes, types et valeurs manquantes.


In [6]:
print("Dimensions du dataset :", df_brut.shape)
print("\nTypes des colonnes :")
display(df_brut.dtypes.to_frame("type"))

missing_values = (
    df_brut.isna().sum()
    .to_frame("nb_valeurs_manquantes")
)
missing_values["pourcentage"] = (missing_values["nb_valeurs_manquantes"] / len(df_brut) * 100).round(2)
missing_values = missing_values.sort_values("nb_valeurs_manquantes", ascending=False)

display(missing_values[missing_values["nb_valeurs_manquantes"] > 0])

Dimensions du dataset : (1500, 15)

Types des colonnes :


,type
trip_id,int64
client_type,object
budget_total,float64
destination,object
saison,object
duree_jours,int64
type_hebergement,object
prix_vol,float64
meteo_prevue,object
activite_principale,object


,nb_valeurs_manquantes,pourcentage
prix_vol,53,3.53
imprevus,53,3.53
activite_principale,48,3.20
budget_total,40,2.67
meteo_prevue,39,2.60
type_hebergement,36,2.40
satisfaction_client,25,1.67
retour_client,25,1.67


### 3. Unicité des séjours

#### Observation métier

Chaque ligne représente un séjour passé. La colonne `trip_id` doit donc identifier un séjour unique.

Un doublon sur `trip_id` pourrait fausser l'analyse car un même séjour serait compté plusieurs fois.


In [7]:
doublons_trip_id = df_brut[df_brut["trip_id"].duplicated(keep=False)].copy()
doublons_lignes = df_brut[df_brut.duplicated(keep=False)].copy()

print("Doublons sur trip_id :")
afficher_cas(doublons_trip_id, colonnes_principales)

print("\nDoublons exacts sur toutes les colonnes :")
afficher_cas(doublons_lignes, colonnes_principales)

Doublons sur trip_id :
Nombre de cas détectés : 0
Aucun cas à afficher.

Doublons exacts sur toutes les colonnes :
Nombre de cas détectés : 0
Aucun cas à afficher.


### 4. Cohérence de la satisfaction client

#### Observation métier

Le cahier des charges indique que `satisfaction_client` est un score de 1 à 5.

Les valeurs inférieures à 1, supérieures à 5 ou manquantes sont donc incohérentes avec la définition officielle de la variable.

Ces cas doivent être corrigés, exclus ou documentés avant d'utiliser `satisfaction_client` comme cible IA.


In [8]:
satisfaction_invalide = df_brut[
    df_brut["satisfaction_client"].isna()
    | ~df_brut["satisfaction_client"].between(1, 5)
].copy()

afficher_cas(
    satisfaction_invalide,
    ["trip_id", "client_type", "destination", "satisfaction_client", "imprevus", "retour_client"]
)

Nombre de cas détectés : 30


,trip_id,client_type,destination,satisfaction_client,imprevus,retour_client
765,766,couple,Bali,NaN,annulation,Une expérience inoubliable.
771,772,solo,Tokyo,NaN,bagages,À éviter.
778,779,couple,Rome,NaN,annulation,Déçu par l'organisation.
800,801,couple,Paris,6.0,aucun,Très bon rapport qualité-prix.
801,802,famille,Dubaï,NaN,aucun,Une expérience inoubliable.
811,812,solo,Rome,6.0,aucun,"Très satisfait, je recommande."
812,813,solo,Bali,0.0,retard_vol,À éviter.
826,827,couple,Sydney,NaN,météo,Déçu par l'organisation.
835,836,famille,Dubaï,NaN,météo,Excellent séjour.
858,859,famille,Paris,NaN,retard_vol,Très mauvaise expérience.


### 5. Prix du vol supérieur au budget total

#### Observation métier

Le `budget_total` représente le budget global du séjour.

Si `prix_vol > budget_total`, le coût du vol dépasse déjà le budget disponible. C'est une incohérence forte ou un séjour impossible à financer sans dépassement.

Le cas est encore plus problématique si `respect_budget = 1`, car cela indique que le budget aurait été respecté malgré un vol supérieur au budget total.


In [9]:
vol_superieur_budget = df_brut[
    df_brut["prix_vol"].notna()
    & df_brut["budget_total"].notna()
    & (df_brut["prix_vol"] > df_brut["budget_total"])
].copy()

vol_superieur_budget["ecart_vol_budget"] = (
    vol_superieur_budget["prix_vol"] - vol_superieur_budget["budget_total"]
).round(2)

print("Tous les cas où le prix du vol dépasse le budget total :")
afficher_cas(
    vol_superieur_budget.sort_values("ecart_vol_budget", ascending=False),
    ["trip_id", "client_type", "destination", "budget_total", "prix_vol", "ecart_vol_budget", "respect_budget"]
)

print("\nCas contradictoires : prix_vol > budget_total ET respect_budget = 1")
vol_superieur_budget_respecte = vol_superieur_budget[vol_superieur_budget["respect_budget"] == 1]
afficher_cas(
    vol_superieur_budget_respecte,
    ["trip_id", "client_type", "destination", "budget_total", "prix_vol", "ecart_vol_budget", "respect_budget"]
)

Tous les cas où le prix du vol dépasse le budget total :
Nombre de cas détectés : 52


,trip_id,client_type,destination,budget_total,prix_vol,ecart_vol_budget,respect_budget
825,826,couple,Rome,500.00,4500.00,4000.00,0
1388,1389,business,Rome,1052.28,4500.00,3447.72,0
1434,1435,business,Paris,500.00,1554.63,1054.63,0
1384,1385,famille,Lisbonne,4152.93,5200.00,1047.07,1
902,903,famille,Rome,500.00,1535.25,1035.25,0
1413,1414,famille,Tokyo,637.34,1593.89,956.55,0
1144,1145,famille,Dubaï,500.00,1443.55,943.55,0
948,949,couple,New York,500.00,1432.13,932.13,0
1402,1403,business,New York,500.00,1426.89,926.89,0
1206,1207,senior,Lisbonne,500.00,1352.70,852.70,0



Cas contradictoires : prix_vol > budget_total ET respect_budget = 1
Nombre de cas détectés : 5


,trip_id,client_type,destination,budget_total,prix_vol,ecart_vol_budget,respect_budget
7,8,couple,Rome,1173.19,1694.67,521.48,1
216,217,business,Rome,1147.25,1438.08,290.83,1
230,231,business,Dubaï,1388.56,1444.72,56.16,1
887,888,solo,Dubaï,380.00,1023.17,643.17,1
1384,1385,famille,Lisbonne,4152.93,5200.00,1047.07,1


### 6. Cohérence entre client business et activité principale

#### Observation métier

Un client `business` peut avoir une activité principale `business`, mais ce n'est pas obligatoire : il peut aussi prolonger son séjour avec de la culture, de la gastronomie ou une activité de loisir.

Ce contrôle n'est donc pas une erreur automatique. C'est un point de vigilance pour vérifier si le dataset décrit bien le comportement attendu des voyageurs professionnels.


In [10]:
client_type_norm = normaliser_texte(df_brut["client_type"])
activite_norm = normaliser_texte(df_brut["activite_principale"])

business_activite_non_business = df_brut[
    (client_type_norm == "business")
    & (activite_norm != "business")
    & (activite_norm != "")
].copy()

non_business_activite_business = df_brut[
    (client_type_norm != "business")
    & (client_type_norm != "")
    & (activite_norm == "business")
].copy()

print("Clients business avec une activité principale non-business :")
afficher_cas(
    business_activite_non_business,
    ["trip_id", "client_type", "destination", "saison", "activite_principale", "budget_total", "satisfaction_client"]
)

print("\nClients non-business avec une activité principale business :")
afficher_cas(
    non_business_activite_business,
    ["trip_id", "client_type", "destination", "saison", "activite_principale", "budget_total", "satisfaction_client"]
)

Clients business avec une activité principale non-business :
Nombre de cas détectés : 200


,trip_id,client_type,destination,saison,activite_principale,budget_total,satisfaction_client
1,2,business,Rome,printemps,randonnée,3288.27,4.0
14,15,business,New York,été,randonnée,12985.57,4.0
19,20,business,Tokyo,printemps,plage,10416.72,3.0
27,28,business,Lisbonne,été,culture,10572.54,5.0
29,30,business,Dubaï,été,culture,2403.24,4.0
34,35,business,Bali,hiver,gastronomie,7427.80,1.0
37,38,business,New York,printemps,plage,13023.12,5.0
40,41,business,Tokyo,été,culture,5227.96,2.0
41,42,business,Tokyo,été,randonnée,5142.10,5.0
44,45,business,Tokyo,été,culture,4100.91,3.0



Clients non-business avec une activité principale business :
Nombre de cas détectés : 122


,trip_id,client_type,destination,saison,activite_principale,budget_total,satisfaction_client
10,11,senior,Bali,automne,business,12845.33,5.0
15,16,solo,Rome,automne,business,6125.88,1.0
16,17,senior,New York,hiver,business,8193.32,2.0
20,21,famille,Bali,automne,business,8186.44,2.0
23,24,solo,Dubaï,automne,business,13391.63,5.0
25,26,couple,Lisbonne,hiver,business,954.98,1.0
42,43,solo,New York,printemps,business,13696.57,2.0
47,48,couple,Lisbonne,hiver,business,9297.44,5.0
71,72,solo,Sydney,hiver,business,2187.78,3.0
129,130,solo,Tokyo,hiver,business,4161.07,3.0


### 7. Activités extérieures et météo risquée

#### Observation métier

L'activité `randonnée` est sensible à la météo.

Si la météo prévue est `pluie` ou `variable`, le séjour peut nécessiter une alternative ou un plan de réorganisation.

Ce n'est pas forcément une erreur, mais c'est une information importante pour l'anticipation des imprévus.


In [11]:
meteo_norm = normaliser_texte(df_brut["meteo_prevue"])
activites_exterieures = ["randonnée", "randonnee"]
meteos_risquees = ["pluie", "variable"]

activites_meteo_risque = df_brut[
    activite_norm.isin(activites_exterieures)
    & meteo_norm.isin(meteos_risquees)
].copy()

afficher_cas(
    activites_meteo_risque,
    ["trip_id", "destination", "saison", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

Nombre de cas détectés : 75


,trip_id,destination,saison,meteo_prevue,activite_principale,imprevus,reorganisation_necessaire,satisfaction_client
1,2,Rome,printemps,variable,randonnée,aucun,0,4.0
7,8,Rome,été,pluie,randonnée,annulation,1,2.0
9,10,New York,automne,variable,randonnée,météo,1,1.0
14,15,New York,été,pluie,randonnée,retard_vol,1,4.0
32,33,Dubaï,printemps,variable,randonnée,bagages,1,1.0
38,39,Lisbonne,été,pluie,randonnée,retard_vol,1,1.0
51,52,Paris,printemps,variable,randonnée,bagages,1,2.0
54,55,Dubaï,automne,variable,randonnée,retard_vol,1,2.0
157,158,Paris,printemps,variable,randonnée,aucun,0,3.0
169,170,Lisbonne,hiver,variable,randonnée,aucun,0,1.0


### 8. Cohérence entre imprévus et réorganisation

#### Observation métier

La variable `reorganisation_necessaire` doit normalement être liée aux imprévus.

Deux situations sont à contrôler :

1. `imprevus = aucun` mais `reorganisation_necessaire = 1` : cas suspect, car il y a une réorganisation sans imprévu déclaré ;
2. `imprevus != aucun` mais `reorganisation_necessaire = 0` : cas possible si l'imprévu est mineur, mais à vérifier.


In [12]:
imprevus_norm = normaliser_texte(df_brut["imprevus"])

aucun_imprevu_mais_reorganisation = df_brut[
    (imprevus_norm == "aucun")
    & (df_brut["reorganisation_necessaire"] == 1)
].copy()

imprevu_sans_reorganisation = df_brut[
    (imprevus_norm != "")
    & (imprevus_norm != "aucun")
    & (df_brut["reorganisation_necessaire"] == 0)
].copy()

print("Aucun imprévu déclaré mais réorganisation nécessaire :")
afficher_cas(
    aucun_imprevu_mais_reorganisation,
    ["trip_id", "destination", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

print("\nImprévu déclaré mais aucune réorganisation :")
afficher_cas(
    imprevu_sans_reorganisation,
    ["trip_id", "destination", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

Aucun imprévu déclaré mais réorganisation nécessaire :
Nombre de cas détectés : 19


,trip_id,destination,meteo_prevue,activite_principale,imprevus,reorganisation_necessaire,satisfaction_client
78,79,Bali,ensoleillé,gastronomie,aucun,1,3.0
135,136,Bali,ensoleillé,plage,aucun,1,3.0
136,137,New York,nuageux,business,aucun,1,4.0
262,263,Lisbonne,pluie,gastronomie,aucun,1,3.0
363,364,Bali,ensoleillé,gastronomie,aucun,1,5.0
395,396,New York,ensoleillé,randonnée,aucun,1,2.0
417,418,Rome,pluie,business,aucun,1,2.0
473,474,Dubaï,pluie,business,aucun,1,2.0
490,491,Lisbonne,nuageux,culture,aucun,1,5.0
496,497,Paris,pluie,plage,aucun,1,2.0



Imprévu déclaré mais aucune réorganisation :
Nombre de cas détectés : 335


,trip_id,destination,meteo_prevue,activite_principale,imprevus,reorganisation_necessaire,satisfaction_client
63,64,Lisbonne,ensoleillé,plage,météo,0,3.0
66,67,Rome,ensoleillé,business,météo,0,4.0
69,70,Tokyo,ensoleillé,randonnée,retard_vol,0,2.0
71,72,Sydney,variable,business,météo,0,3.0
72,73,Lisbonne,ensoleillé,gastronomie,retard_vol,0,1.0
79,80,Tokyo,nuageux,randonnée,bagages,0,2.0
81,82,New York,variable,gastronomie,météo,0,3.0
86,87,Dubaï,pluie,culture,retard_vol,0,1.0
90,91,Paris,ensoleillé,culture,annulation,0,3.0
91,92,Bali,variable,culture,retard_vol,0,4.0


### 9. Imprévus et satisfaction faible

#### Observation métier

Un imprévu peut réduire la satisfaction client.

Les cas où `imprevus != aucun` et `satisfaction_client <= 2` sont cohérents métier, mais importants pour comprendre les facteurs d'insatisfaction.

Ils peuvent alimenter l'analyse explicative, mais il faut éviter d'utiliser les imprévus comme variable d'entrée si l'objectif est de prédire la satisfaction avant le départ.


In [13]:
imprevus_satisfaction_faible = df_brut[
    (imprevus_norm != "")
    & (imprevus_norm != "aucun")
    & df_brut["satisfaction_client"].notna()
    & (df_brut["satisfaction_client"] <= 2)
].copy()

afficher_cas(
    imprevus_satisfaction_faible,
    ["trip_id", "client_type", "destination", "imprevus", "reorganisation_necessaire", "respect_budget", "satisfaction_client", "retour_client"]
)

Nombre de cas détectés : 573


,trip_id,client_type,destination,imprevus,reorganisation_necessaire,respect_budget,satisfaction_client,retour_client
4,5,solo,Paris,retard_vol,1,1,2.0,Séjour mitigé.
7,8,couple,Rome,annulation,1,1,2.0,Séjour mitigé.
8,9,couple,Paris,annulation,1,0,2.0,Séjour mitigé.
9,10,solo,New York,météo,1,0,1.0,Séjour mitigé.
11,12,senior,Paris,retard_vol,1,0,2.0,Séjour mitigé.
12,13,famille,New York,météo,1,1,1.0,Séjour mitigé.
13,14,solo,Sydney,annulation,1,1,2.0,Séjour mitigé.
22,23,famille,Lisbonne,bagages,1,0,1.0,Séjour mitigé.
25,26,couple,Lisbonne,retard_vol,1,1,1.0,Séjour mitigé.
28,29,solo,Bali,retard_vol,1,0,2.0,Séjour mitigé.


### 10. Risque de fuite de données pour la modélisation

#### Observation métier

Certaines colonnes décrivent le résultat du séjour après coup :

- `imprevus` ;
- `reorganisation_necessaire` ;
- `respect_budget` ;
- `retour_client`.

Si le modèle doit prédire la satisfaction avant le départ, ces variables ne doivent pas être utilisées comme entrées, car elles ne sont pas connues au moment de la recommandation.

Elles peuvent cependant servir à une analyse après séjour ou à un modèle secondaire d'amélioration continue.


In [14]:
colonnes_post_sejour = ["imprevus", "reorganisation_necessaire", "respect_budget", "retour_client", "satisfaction_client"]
display(df_brut[["trip_id", *colonnes_post_sejour]].head(10))

,trip_id,imprevus,reorganisation_necessaire,respect_budget,retour_client,satisfaction_client
0,1,annulation,1,0,Séjour mitigé.,3.0
1,2,aucun,0,1,Excellent séjour.,4.0
2,3,météo,1,1,Excellent séjour.,5.0
3,4,météo,1,1,Séjour mitigé.,3.0
4,5,retard_vol,1,1,Séjour mitigé.,2.0
5,6,météo,1,1,Excellent séjour.,5.0
6,7,météo,1,1,Excellent séjour.,5.0
7,8,annulation,1,1,Séjour mitigé.,2.0
8,9,annulation,1,0,Séjour mitigé.,2.0
9,10,météo,1,0,Séjour mitigé.,1.0


### 11. Synthèse des contrôles

Ce tableau résume les contrôles réalisés et le nombre de cas détectés.

Il sert de base pour décider quoi corriger, quoi conserver et quoi documenter avant la modélisation.


In [15]:
synthese_controles = pd.DataFrame([
    {
        "controle": "Doublons trip_id",
        "nb_cas": len(doublons_trip_id),
        "niveau": "Incohérence forte si > 0",
        "action_recommandee": "Supprimer ou fusionner les doublons"
    },
    {
        "controle": "Satisfaction hors échelle ou manquante",
        "nb_cas": len(satisfaction_invalide),
        "niveau": "Incohérence forte",
        "action_recommandee": "Corriger, exclure ou imputer selon justification"
    },
    {
        "controle": "Prix du vol supérieur au budget total",
        "nb_cas": len(vol_superieur_budget),
        "niveau": "Incohérence forte",
        "action_recommandee": "Contrôler budget_total, prix_vol et respect_budget"
    },
    {
        "controle": "Prix du vol > budget total et respect_budget = 1",
        "nb_cas": len(vol_superieur_budget_respecte),
        "niveau": "Contradiction métier",
        "action_recommandee": "Corriger respect_budget ou les montants"
    },
    {
        "controle": "Client business avec activité non-business",
        "nb_cas": len(business_activite_non_business),
        "niveau": "Point de vigilance",
        "action_recommandee": "Vérifier si le séjour mixte business/loisir est attendu"
    },
    {
        "controle": "Client non-business avec activité business",
        "nb_cas": len(non_business_activite_business),
        "niveau": "Point de vigilance",
        "action_recommandee": "Vérifier la définition de l'activité principale"
    },
    {
        "controle": "Randonnée avec météo risquée",
        "nb_cas": len(activites_meteo_risque),
        "niveau": "Signal métier utile",
        "action_recommandee": "Créer une variable meteo_risque ciblée sur la randonnée"
    },
    {
        "controle": "Aucun imprévu mais réorganisation nécessaire",
        "nb_cas": len(aucun_imprevu_mais_reorganisation),
        "niveau": "Cas suspect",
        "action_recommandee": "Contrôler la cohérence entre imprevus et reorganisation_necessaire"
    },
    {
        "controle": "Imprévu déclaré sans réorganisation",
        "nb_cas": len(imprevu_sans_reorganisation),
        "niveau": "Point de vigilance",
        "action_recommandee": "Conserver si imprévu mineur, sinon corriger"
    },
    {
        "controle": "Imprévu avec satisfaction faible",
        "nb_cas": len(imprevus_satisfaction_faible),
        "niveau": "Signal explicatif",
        "action_recommandee": "Utiliser pour comprendre l'insatisfaction, attention à la fuite de données"
    },
])

display(synthese_controles)

,controle,nb_cas,niveau,action_recommandee
0,Doublons trip_id,0,Incohérence forte si > 0,Supprimer ou fusionner les doublons
1,Satisfaction hors échelle ou manquante,30,Incohérence forte,"Corriger, exclure ou imputer selon justification"
2,Prix du vol supérieur au budget total,52,Incohérence forte,"Contrôler budget_total, prix_vol et respect_bu..."
3,Prix du vol > budget total et respect_budget = 1,5,Contradiction métier,Corriger respect_budget ou les montants
4,Client business avec activité non-business,200,Point de vigilance,Vérifier si le séjour mixte business/loisir es...
5,Client non-business avec activité business,122,Point de vigilance,Vérifier la définition de l'activité principale
6,Randonnée avec météo risquée,75,Signal métier utile,Créer une variable meteo_risque ciblée sur la ...
7,Aucun imprévu mais réorganisation nécessaire,19,Cas suspect,Contrôler la cohérence entre imprevus et reorg...
8,Imprévu déclaré sans réorganisation,335,Point de vigilance,"Conserver si imprévu mineur, sinon corriger"
9,Imprévu avec satisfaction faible,573,Signal explicatif,"Utiliser pour comprendre l'insatisfaction, att..."


### 12. Sélection des incohérences à traiter avant modélisation

Toutes les observations précédentes ne doivent pas être corrigées. Certaines sont des signaux métier utiles, d'autres sont des incohérences qui peuvent dégrader directement le modèle.

Pour le modèle principal envisagé, la cible est `satisfaction_client`. On distingue donc :

- les problèmes à traiter obligatoirement avant l'entraînement ;
- les variables à exclure pour éviter la fuite de données ;
- les points de vigilance à conserver ou transformer en variables métier.


In [16]:

selection_traitement_modele = pd.DataFrame([
    {
        "controle": "Satisfaction hors ?chelle ou manquante",
        "impact_modele": "Impact critique sur la cible y",
        "decision": "Traiter obligatoirement",
        "traitement_recommande": "Supprimer les lignes sans cible ou hors ?chelle 1-5, sauf justification m?tier de recodage",
    },
    {
        "controle": "Prix du vol sup?rieur au budget total",
        "impact_modele": "Cr?e des relations budg?taires impossibles et fausse les ratios",
        "decision": "Traiter obligatoirement",
        "traitement_recommande": "Exclure les lignes concern?es, car corriger prix_vol ou budget_total inventerait une information",
    },
    {
        "controle": "Prix du vol > budget total et respect_budget = 1",
        "impact_modele": "Contradiction m?tier forte entre budget et r?sultat",
        "decision": "Trait? automatiquement",
        "traitement_recommande": "Ces lignes sont incluses dans la r?gle g?n?rale prix_vol > budget_total et sont donc supprim?es",
    },
    {
        "controle": "Aucun impr?vu mais r?organisation n?cessaire",
        "impact_modele": "Contradiction op?rationnelle entre incident d?clar? et r?organisation",
        "decision": "Supprimer en nettoyage strict",
        "traitement_recommande": "Exclure ces lignes car une r?organisation sans impr?vu d?clar? rend l'historique incoh?rent",
    },
    {
        "controle": "Impr?vu d?clar? sans r?organisation",
        "impact_modele": "Contradiction op?rationnelle si l'on consid?re tout impr?vu comme n?cessitant une action",
        "decision": "Supprimer en nettoyage strict",
        "traitement_recommande": "Exclure ces lignes dans cette version stricte du dataset ; documenter que ce choix r?duit le volume disponible",
    },
    {
        "controle": "Variables post-s?jour : imprevus, reorganisation_necessaire, respect_budget, retour_client",
        "impact_modele": "Risque majeur de fuite de donn?es si la pr?diction est faite avant le d?part",
        "decision": "Exclure des features du mod?le pr?-voyage ; autoriser selon l'objectif post-voyage",
        "traitement_recommande": "Ne pas les mettre dans X pr?-voyage ; les utiliser seulement dans le mod?le post-voyage explicatif valid?",
    },
])

display(selection_traitement_modele)


,controle,impact_modele,decision,traitement_recommande
0,Satisfaction hors ?chelle ou manquante,Impact critique sur la cible y,Traiter obligatoirement,Supprimer les lignes sans cible ou hors ?chell...
1,Prix du vol sup?rieur au budget total,Cr?e des relations budg?taires impossibles et ...,Traiter obligatoirement,"Exclure les lignes concern?es, car corriger pr..."
2,Prix du vol > budget total et respect_budget = 1,Contradiction m?tier forte entre budget et r?s...,Trait? automatiquement,Ces lignes sont incluses dans la r?gle g?n?ral...
3,Aucun impr?vu mais r?organisation n?cessaire,Contradiction op?rationnelle entre incident d?...,Supprimer en nettoyage strict,Exclure ces lignes car une r?organisation sans...
4,Impr?vu d?clar? sans r?organisation,Contradiction op?rationnelle si l'on consid?re...,Supprimer en nettoyage strict,Exclure ces lignes dans cette version stricte ...
5,"Variables post-s?jour : imprevus, reorganisati...",Risque majeur de fuite de donn?es si la pr?dic...,Exclure des features du mod?le pr?-voyage ; au...,Ne pas les mettre dans X pr?-voyage ; les util...



#### Observations non trait?es comme erreurs

Les contr?les suivants ne sont pas corrig?s automatiquement car ils peuvent repr?senter des comportements m?tier r?els ou des signaux utiles :

- `client business` avec activit? non-business : possible s?jour mixte business/loisir ;
- `client non-business` avec activit? business : possible voyage personnel avec objectif professionnel ;
- activit? ext?rieure avec m?t?o risqu?e : signal utile pour cr?er `meteo_risque`, pas une erreur ;
- impr?vu avec satisfaction faible : signal explicatif, pas une incoh?rence.

Les contradictions entre `imprevus` et `reorganisation_necessaire` sont d?sormais supprim?es dans le nettoyage strict du dataset. Cette r?gle est appliqu?e une seule fois sur `df_model`, donc elle concerne ? la fois la mod?lisation pr?-voyage et la mod?lisation post-voyage.


### 13. Conclusion métier

Le dataset est exploitable pour un prototype IA, mais plusieurs points doivent être documentés avant la modélisation :

- la cible `satisfaction_client` doit être nettoyée car certaines valeurs ne respectent pas l'échelle 1 à 5 ;
- les contradictions fortes liées au budget doivent être traitées ;
- les variables post-séjour doivent être séparées des variables disponibles avant le départ pour éviter la fuite de données ;
- les contrôles météo, activité, budget journalier et type client peuvent devenir des variables utiles de feature engineering.

Ces observations justifient une étape de préparation des données avant la création du modèle IA.


## 6. Contrôles de cohérence métier initiaux


In [ ]:

imprevus_initiaux_norm = df_raw["imprevus"].astype("string").str.strip().str.lower()

controle_initial = pd.DataFrame([
    {
        "controle": "doublons trip_id",
        "nb_lignes": int(df_raw["trip_id"].duplicated().sum()),
        "impact": "risque de double comptage",
    },
    {
        "controle": "satisfaction manquante ou hors ?chelle 1-5",
        "nb_lignes": int((df_raw[TARGET_COLUMN].isna() | ~df_raw[TARGET_COLUMN].between(1, 5)).sum()),
        "impact": "cible invalide pour l'apprentissage",
    },
    {
        "controle": "prix_vol > budget_total",
        "nb_lignes": int((df_raw["prix_vol"] > df_raw["budget_total"]).sum()),
        "impact": "incoh?rence budg?taire m?tier",
    },
    {
        "controle": "prix_vol > budget_total et respect_budget = 1",
        "nb_lignes": int(((df_raw["prix_vol"] > df_raw["budget_total"]) & (df_raw["respect_budget"] == 1)).sum()),
        "impact": "contradiction m?tier forte incluse dans la r?gle budget",
    },
    {
        "controle": "aucun impr?vu mais r?organisation n?cessaire",
        "nb_lignes": int(((imprevus_initiaux_norm == "aucun") & (df_raw["reorganisation_necessaire"] == 1)).sum()),
        "impact": "contradiction op?rationnelle",
    },
    {
        "controle": "impr?vu d?clar? sans r?organisation",
        "nb_lignes": int(((imprevus_initiaux_norm != "") & (imprevus_initiaux_norm != "aucun") & (df_raw["reorganisation_necessaire"] == 0)).sum()),
        "impact": "contradiction op?rationnelle en nettoyage strict",
    },
])

display(controle_initial)



### 6.1 Contr?le des valeurs n?gatives

Les colonnes num?riques m?tier suivantes ne doivent pas contenir de valeurs n?gatives : `budget_total`, `duree_jours` et `prix_vol`.

Une valeur n?gative serait incoh?rente pour le projet : un budget, une dur?e de s?jour ou un prix de vol ne peut pas ?tre inf?rieur ? z?ro. Ce contr?le est effectu? avant le nettoyage afin d'identifier les anomalies fortes du dataset brut.


In [ ]:

colonnes_controle_negatif = ["budget_total", "duree_jours", "prix_vol"]

controle_valeurs_negatives = []
cas_valeurs_negatives = []

for column in colonnes_controle_negatif:
    serie_numerique = pd.to_numeric(df_raw[column], errors="coerce")
    mask_negatif = serie_numerique < 0
    nb_negatifs = int(mask_negatif.sum())

    controle_valeurs_negatives.append({
        "colonne": column,
        "nb_valeurs_negatives": nb_negatifs,
        "minimum_observe": serie_numerique.min(),
        "decision": "? supprimer ou corriger" if nb_negatifs > 0 else "aucune action n?cessaire",
    })

    if nb_negatifs > 0:
        cas = df_raw.loc[mask_negatif, ["trip_id", column]].copy()
        cas["colonne_negative"] = column
        cas_valeurs_negatives.append(cas)

controle_valeurs_negatives = pd.DataFrame(controle_valeurs_negatives)
display(controle_valeurs_negatives)

if cas_valeurs_negatives:
    cas_valeurs_negatives = pd.concat(cas_valeurs_negatives, ignore_index=True)
    display(cas_valeurs_negatives)
else:
    print("Aucune valeur n?gative d?tect?e dans les colonnes num?riques contr?l?es.")


## 7. Nettoyage des donnees



#### Integrite, formats et corrections appliquees au nettoyage

- Les noms de colonnes du dataset sont conserves car ils sont explicites et coherents avec le metier : `budget_total`, `duree_jours`, `prix_vol`, `satisfaction_client`, `respect_budget`, etc.
- Le format des donnees est adapte a l'usage : les colonnes numeriques sont converties avec `pd.to_numeric`, la cible est convertie en entier et les colonnes texte sont standardisees en minuscules avec suppression des espaces inutiles.
- Les donnees alterees ou incoherentes sont traitees avant modelisation : suppression des cibles invalides, suppression des cas `prix_vol > budget_total` et suppression des contradictions strictes entre `imprevus` et `reorganisation_necessaire`.
- Les traitements qui apprennent des parametres statistiques ne sont plus appliques globalement avant le split : imputation, remplacement des outliers IQR, standardisation et encodage sont integres dans le pipeline `scikit-learn`.
- Les parametres d'imputation, de traitement IQR, de normalisation et d'encodage sont donc appris uniquement sur le jeu d'entrainement, puis appliques au jeu de test.
- Les traitements sont documentes par `cleaning_report`, `incoherence_metier_report` et `pipeline_treatment_report`, affiches juste apres le nettoyage.


In [ ]:

def nettoyer_dataset(df_source: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = df_source.copy()
    nb_initial = len(df)

    for column in df.select_dtypes(include=["object", "string"]).columns:
        cleaned_column = (
            df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace({"": np.nan, "nan": np.nan})
        )
        df[column] = cleaned_column.mask(cleaned_column.isna(), np.nan).astype(object)

    numeric_source_columns = [
        "budget_total",
        "duree_jours",
        "prix_vol",
        TARGET_COLUMN,
        "reorganisation_necessaire",
        "respect_budget",
    ]

    for column in numeric_source_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # 1. Cible invalide : suppression obligatoire car satisfaction_client est y.
    df = df[df[TARGET_COLUMN].between(1, 5)].copy()
    nb_after_target = len(df)
    df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

    # 2. Incoherence budgetaire : si les deux valeurs sont connues,
    # le vol ne peut pas depasser le budget total.
    budget_valid_mask = (
        df["prix_vol"].isna()
        | df["budget_total"].isna()
        | (df["prix_vol"] <= df["budget_total"])
    )
    df = df[budget_valid_mask].copy()
    nb_after_budget = len(df)

    # 3. Incoherences strictes issues de l'analyse metier.
    # Les simples signaux metier ne sont pas supprimes ici.
    imprevus_norm = df["imprevus"].fillna("").astype("string").str.strip().str.lower()

    aucun_imprevu_mais_reorganisation_mask = (
        (imprevus_norm == "aucun")
        & (df["reorganisation_necessaire"] == 1)
    )
    imprevu_sans_reorganisation_mask = (
        (imprevus_norm != "")
        & (imprevus_norm != "aucun")
        & (df["reorganisation_necessaire"] == 0)
    )

    incoherence_metier_report = pd.DataFrame([
        {
            "incoherence": "aucun imprevu mais reorganisation necessaire",
            "nb_lignes_supprimees": int(aucun_imprevu_mais_reorganisation_mask.sum()),
            "raison": "contradiction entre absence d'incident declare et besoin de reorganisation",
        },
        {
            "incoherence": "imprevu declare sans reorganisation",
            "nb_lignes_supprimees": int(imprevu_sans_reorganisation_mask.sum()),
            "raison": "contradiction operationnelle retenue dans le nettoyage strict du dataset final",
        },
    ])

    incoherence_metier_mask = (
        aucun_imprevu_mais_reorganisation_mask
        | imprevu_sans_reorganisation_mask
    )
    df = df[~incoherence_metier_mask].copy()
    nb_after_incoherences_metier = len(df)

    # Regles metier sans apprentissage statistique.
    # L'imputation statistique reste geree plus tard par le pipeline sklearn.
    df["imprevus"] = df["imprevus"].fillna("aucun").replace({"nan": "aucun"})
    df["retour_client"] = df["retour_client"].fillna("").replace({"nan": ""})

    cleaning_report = pd.DataFrame([
        {
            "etape": "dataset brut",
            "nb_lignes": nb_initial,
            "lignes_supprimees": 0,
        },
        {
            "etape": "cible satisfaction_client valide",
            "nb_lignes": nb_after_target,
            "lignes_supprimees": nb_initial - nb_after_target,
        },
        {
            "etape": "coherence initiale prix_vol <= budget_total",
            "nb_lignes": nb_after_budget,
            "lignes_supprimees": nb_after_target - nb_after_budget,
        },
        {
            "etape": "coherence stricte imprevus / reorganisation",
            "nb_lignes": nb_after_incoherences_metier,
            "lignes_supprimees": nb_after_budget - nb_after_incoherences_metier,
        },
    ])

    pipeline_treatment_report = pd.DataFrame([
        {
            "traitement": "imputation numerique",
            "moment": "pipeline apres train/test split",
            "parametres_appris": "mediane calculee uniquement sur le train",
        },
        {
            "traitement": "outliers IQR numeriques continus",
            "moment": "pipeline apres train/test split",
            "parametres_appris": "Q1, Q3, bornes IQR et mediane calcules uniquement sur le train",
        },
        {
            "traitement": "standardisation numerique",
            "moment": "pipeline apres train/test split",
            "parametres_appris": "moyenne et ecart-type calcules uniquement sur le train",
        },
        {
            "traitement": "imputation categorielle et encodage OneHot",
            "moment": "pipeline apres train/test split",
            "parametres_appris": "mode et categories apprises uniquement sur le train",
        },
        {
            "traitement": "SMOTE si experience activee",
            "moment": "ImbPipeline sur train uniquement",
            "parametres_appris": "reequilibrage applique uniquement aux donnees d'entrainement ou aux folds train",
        },
    ])

    return df, cleaning_report, incoherence_metier_report, pipeline_treatment_report


df_model, cleaning_report, incoherence_metier_report, pipeline_treatment_report = nettoyer_dataset(df_raw)

display(cleaning_report)
display(incoherence_metier_report)
display(pipeline_treatment_report)
print(f"Volume final apres nettoyage metier : {len(df_model)} lignes")
print(f"Valeurs manquantes conservees avant pipeline : {int(df_model.isna().sum().sum())}")


### 7.1 Équilibre des classes après nettoyage

Cette vérification est réalisée juste après le nettoyage des données, avant le feature engineering. Elle permet de contrôler la distribution réelle de la cible dans le dataset nettoyé qui servira ensuite de base à la modélisation.

On observe deux distributions :

- `satisfaction_client` en 5 classes, correspondant aux notes originales de 1 à 5 ;
- `satisfaction_client` regroup?e en 3 classes pour les mod?les pr?-voyage et post-voyage.


In [ ]:
def satisfaction_3_classes_apres_nettoyage(value: int) -> int:
    if value <= 2:
        return 0
    if value == 3:
        return 1
    return 2


libelles_5_classes_apres_nettoyage = {
    1: "1_tres_insatisfait",
    2: "2_insatisfait",
    3: "3_neutre",
    4: "4_satisfait",
    5: "5_tres_satisfait",
}

libelles_3_classes_apres_nettoyage = {
    0: "insatisfait_1_2",
    1: "neutre_3",
    2: "satisfait_4_5",
}


def calculer_distribution_apres_nettoyage(y: pd.Series, libelles: dict[int, str], cible: str) -> pd.DataFrame:
    distribution = (
        y.value_counts()
        .sort_index()
        .rename_axis("classe")
        .reset_index(name="nombre")
    )
    distribution["libelle"] = distribution["classe"].map(libelles)
    distribution["pourcentage"] = (
        distribution["nombre"] / distribution["nombre"].sum() * 100
    ).round(2)
    distribution["cible"] = cible
    return distribution[["cible", "classe", "libelle", "nombre", "pourcentage"]]


distribution_nettoyage_5_classes = calculer_distribution_apres_nettoyage(
    df_model[TARGET_COLUMN].astype(int),
    libelles_5_classes_apres_nettoyage,
    "satisfaction_client_5_classes_apres_nettoyage",
)

distribution_nettoyage_3_classes = calculer_distribution_apres_nettoyage(
    df_model[TARGET_COLUMN].astype(int).apply(satisfaction_3_classes_apres_nettoyage),
    libelles_3_classes_apres_nettoyage,
    "satisfaction_client_3_classes_apres_nettoyage",
)

print("Équilibre des classes après nettoyage")
display(distribution_nettoyage_5_classes)
display(distribution_nettoyage_3_classes)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(
    distribution_nettoyage_5_classes["libelle"],
    distribution_nettoyage_5_classes["pourcentage"],
    color="#4c78a8",
)
axes[0].set_title("Après nettoyage - 5 classes")
axes[0].set_ylabel("Pourcentage des lignes (%)")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    distribution_nettoyage_3_classes["libelle"],
    distribution_nettoyage_3_classes["pourcentage"],
    color=["#d95f02", "#7570b3", "#1b9e77"],
)
axes[1].set_title("Après nettoyage - 3 classes")
axes[1].set_ylabel("Pourcentage des lignes (%)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Feature engineering


#### Pertinence metier des variables preparees

- Les nouvelles variables sont nommees de facon descriptive : `budget_par_jour`, `part_vol_budget`, `sejour_long`, `meteo_risque`, `hebergement_luxe`, `gravite_imprevu`.
- Les variables derivees renforcent la pertinence metier du dataset : elles rendent explicites des notions utiles comme le poids du vol dans le budget, le risque meteo, la duree du sejour ou le niveau de complexite operationnelle.
- Les variables post-voyage explicatives sont clairement separees des variables pre-voyage pour eviter la fuite de donnees.
- Le dataset final `df_model` est controle avant la separation train/test : dimensions, exemples de variables creees et equilibre des classes.


In [ ]:

def ajouter_features_base(df_source: pd.DataFrame) -> pd.DataFrame:
    df = df_source.copy()

    safe_duree = df["duree_jours"].replace(0, np.nan)
    safe_budget = df["budget_total"].replace(0, np.nan)

    def indicateur(condition: pd.Series, missing_mask: pd.Series) -> pd.Series:
        return pd.Series(
            np.where(missing_mask, np.nan, condition.astype(int)),
            index=df.index,
        )

    df["budget_par_jour"] = df["budget_total"] / safe_duree
    df["part_vol_budget"] = df["prix_vol"] / safe_budget
    df["budget_hors_vol"] = df["budget_total"] - df["prix_vol"]
    df["sejour_long"] = indicateur(df["duree_jours"] >= 14, df["duree_jours"].isna())
    df["meteo_risque"] = indicateur(
        df["meteo_prevue"].isin(["pluie", "variable"]),
        df["meteo_prevue"].isna(),
    )
    df["randonnee_meteo_risque"] = indicateur(
        df["activite_principale"].isin(["randonn?e", "randonnee", "randonn?e"])
        & df["meteo_prevue"].isin(["pluie", "variable"]),
        df["activite_principale"].isna() | df["meteo_prevue"].isna(),
    )
    df["saison_haute"] = indicateur(
        df["saison"].isin(["?t?", "ete", "?t?", "hiver"]),
        df["saison"].isna(),
    )
    df["client_business"] = indicateur(
        df["client_type"] == "business",
        df["client_type"].isna(),
    )
    df["hebergement_luxe"] = indicateur(
        df["type_hebergement"].isin(["resort", "villa"]),
        df["type_hebergement"].isna(),
    )

    destination_enrichment = pd.DataFrame({
        "destination": ["paris", "rome", "lisbonne", "new york", "duba?", "tokyo", "bali", "sydney"],
        "region_destination": ["europe", "europe", "europe", "am?rique du nord", "moyen-orient", "asie", "asie", "oc?anie"],
        "type_destination": ["culture", "culture", "culture", "urbain_business", "luxe_shopping", "culture_urbain", "plage_luxe", "urbain_nature"],
        "distance_vol_categorie": ["court", "court", "court", "long", "moyen", "long", "long", "long"],
        "cout_vie_destination": ["?lev?", "moyen", "moyen", "?lev?", "?lev?", "?lev?", "moyen", "?lev?"],
        "destination_luxe": [1, 1, 0, 1, 1, 1, 1, 1],
        "decalage_horaire_categorie": ["faible", "faible", "faible", "moyen", "moyen", "fort", "fort", "fort"],
        "risque_meteo_destination": ["moyen", "moyen", "faible", "moyen", "faible", "moyen", "?lev?", "moyen"],
    })

    df = df.merge(destination_enrichment, on="destination", how="left", validate="many_to_one")

    for column in ["region_destination", "distance_vol_categorie"]:
        if df[column].isna().any():
            df[column] = df[column].fillna("inconnu")

    # Variables post-voyage explicatives : utilis?es uniquement dans le mod?le post-voyage.
    df["imprevu_present"] = indicateur(df["imprevus"] != "aucun", df["imprevus"].isna())
    df["imprevu_transport"] = indicateur(
        df["imprevus"].isin(["retard_vol", "annulation", "bagages"]),
        df["imprevus"].isna(),
    )
    df["imprevu_meteo"] = indicateur(
        df["imprevus"].isin(["m?t?o", "meteo", "m?t?o"]),
        df["imprevus"].isna(),
    )
    df["budget_non_respecte"] = indicateur(
        df["respect_budget"] == 0,
        df["respect_budget"].isna(),
    )
    df["gravite_imprevu"] = df["imprevus"].map({
        "aucun": 0,
        "m?t?o": 1,
        "meteo": 1,
        "m?t?o": 1,
        "bagages": 1,
        "retard_vol": 2,
        "annulation": 3,
    })
    df["annulation_et_reorganisation"] = indicateur(
        (df["imprevus"] == "annulation")
        & (df["reorganisation_necessaire"] == 1),
        df["imprevus"].isna() | df["reorganisation_necessaire"].isna(),
    )
    df["retard_et_budget_non_respecte"] = indicateur(
        (df["imprevus"] == "retard_vol")
        & (df["budget_non_respecte"] == 1),
        df["imprevus"].isna() | df["budget_non_respecte"].isna(),
    )
    df["imprevu_transport_et_sejour_court"] = indicateur(
        (df["imprevu_transport"] == 1)
        & (df["duree_jours"] <= 5),
        df["imprevu_transport"].isna() | df["duree_jours"].isna(),
    )
    df["budget_tendu"] = indicateur(
        df["part_vol_budget"] >= 0.5,
        df["part_vol_budget"].isna(),
    )
    df["budget_tendu_et_hebergement_luxe"] = indicateur(
        (df["budget_tendu"] == 1)
        & (df["hebergement_luxe"] == 1),
        df["budget_tendu"].isna() | df["hebergement_luxe"].isna(),
    )

    for column in ["budget_par_jour", "part_vol_budget", "budget_hors_vol"]:
        df[column] = df[column].replace([np.inf, -np.inf], np.nan)

    for column in df.select_dtypes(include=["object", "string"]).columns:
        df[column] = df[column].mask(df[column].isna(), np.nan).astype(object)

    return df


df_model = ajouter_features_base(df_model)

features_supprimees_modelisation = [
    "budget_hors_vol",
    "saison_haute",
    "cout_vie_destination",
    "type_destination",
    "decalage_horaire_categorie",
    "risque_meteo_destination",
    "annulation_et_reorganisation",
    "retard_et_budget_non_respecte",
    "imprevu_transport_et_sejour_court",
    "budget_tendu_et_hebergement_luxe",
]

df_model = df_model.drop(columns=features_supprimees_modelisation, errors="ignore")

features_base = [
    "budget_par_jour",
    "part_vol_budget",
    "sejour_long",
    "meteo_risque",
    "randonnee_meteo_risque",
    "client_business",
    "hebergement_luxe",
    "destination_luxe",
]

features_post_voyage_explicatives = [
    "imprevus",
    "reorganisation_necessaire",
    "respect_budget",
    "imprevu_present",
    "imprevu_transport",
    "imprevu_meteo",
    "budget_non_respecte",
    "budget_tendu",
    "gravite_imprevu",
]

print("Dimensions apr?s feature engineering :", df_model.shape)
display(df_model[features_base + [TARGET_COLUMN]].head())
print(f"Valeurs manquantes avant pipeline : {int(df_model.isna().sum().sum())}")


### 8.1 Équilibre des classes du dataset final de modélisation

Cette vérification est réalisée sur `df_model`, c'est-à-dire le dataset final après nettoyage, traitement des incohérences et feature engineering. Elle permet de contrôler la distribution de la cible avant toute séparation train/test.

Deux lectures sont utiles :

- la distribution originale de `satisfaction_client` sur les 5 notes `1` ? `5`, conserv?e pour analyse descriptive ;
- la distribution regroup?e en 3 classes, utilis?e pour les mod?les pr?-voyage et post-voyage.


In [ ]:
def regrouper_satisfaction_3_classes(value: int) -> int:
    if value <= 2:
        return 0
    if value == 3:
        return 1
    return 2


libelles_satisfaction_5 = {
    1: "1_tres_insatisfait",
    2: "2_insatisfait",
    3: "3_neutre",
    4: "4_satisfait",
    5: "5_tres_satisfait",
}

libelles_satisfaction_3 = {
    0: "insatisfait_1_2",
    1: "neutre_3",
    2: "satisfait_4_5",
}


def distribution_classes(y: pd.Series, libelles: dict[int, str], nom_cible: str) -> pd.DataFrame:
    distribution = (
        y.value_counts()
        .sort_index()
        .rename_axis("classe")
        .reset_index(name="nombre")
    )
    distribution["libelle"] = distribution["classe"].map(libelles)
    distribution["pourcentage"] = (
        distribution["nombre"] / distribution["nombre"].sum() * 100
    ).round(2)
    distribution["cible"] = nom_cible
    return distribution[["cible", "classe", "libelle", "nombre", "pourcentage"]]


distribution_satisfaction_5 = distribution_classes(
    df_model[TARGET_COLUMN].astype(int),
    libelles_satisfaction_5,
    "satisfaction_client_5_classes",
)

distribution_satisfaction_3 = distribution_classes(
    df_model[TARGET_COLUMN].astype(int).apply(regrouper_satisfaction_3_classes),
    libelles_satisfaction_3,
    "satisfaction_client_3_classes",
)

print("Distribution de la cible sur le dataset final df_model")
display(distribution_satisfaction_5)
display(distribution_satisfaction_3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(
    distribution_satisfaction_5["libelle"],
    distribution_satisfaction_5["pourcentage"],
    color="#4c78a8",
)
axes[0].set_title("Satisfaction client - 5 classes")
axes[0].set_ylabel("Pourcentage des lignes (%)")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    distribution_satisfaction_3["libelle"],
    distribution_satisfaction_3["pourcentage"],
    color=["#d95f02", "#7570b3", "#1b9e77"],
)
axes[1].set_title("Satisfaction client - 3 classes")
axes[1].set_ylabel("Pourcentage des lignes (%)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()



### 8.2 Stockage, cycle de vie et gouvernance du dataset

Cette section complete la preparation des donnees avec les choix de stockage, le cycle de vie du dataset et les validations attendues par les parties prenantes.

#### Choix du modele de stockage

| Element | Choix retenu | Justification |
| --- | --- | --- |
| Dataset brut | Fichier CSV dans `data/` | Format simple, lisible, adapte a un dataset tabulaire de faible volume |
| Dataset prepare | `df_model` reconstruit dans le notebook | Evite de multiplier les fichiers derives ; garantit la reproductibilite des transformations metier |
| Pretraitements statistiques | Pipeline `scikit-learn` | Imputation, outliers IQR, standardisation et encodage sont appris sur le train uniquement |
| Experimentations | Notebook final + MLflow local | Le notebook documente la logique ; MLflow trace les runs, parametres et metriques |
| Artefacts lourds | Non versionnes dans Git | `mlflow.db`, `mlruns/`, `mlartifacts/`, `logs/` sont exclus du depot pour eviter d'alourdir Git |
| Deploiement futur | A definir apres validation du notebook final | Un stockage versionne des donnees preparees et un registre de modeles seraient necessaires en production |

Le choix du CSV local est coherent pour le prototype actuel. Il serait insuffisant pour une exploitation multi-utilisateurs, automatisee ou connectee a des donnees clients reelles.

#### Cycle de vie du dataset dans ce projet

| Etape | Objet produit | Verification realisee |
| --- | --- | --- |
| 1. Donnees brutes | `data/Examen_travel_planning_dataset.csv` | Existence, chargement, dimensions, types, valeurs manquantes, doublons, incoherences metier |
| 2. Nettoyage metier | `df_model` apres `nettoyer_dataset(df_raw)` | Cible valide 1-5, suppression des incoherences fortes dont budget et imprevus/reorganisation |
| 3. Controle apres nettoyage | Distributions de classes apres nettoyage | Verification de l'equilibre de la cible avant feature engineering |
| 4. Feature engineering | Variables derivees ajoutees a `df_model` | Creation de variables metier utiles et suppression des features non retenues |
| 5. Train/test split | `X_train`, `X_test`, `y_train`, `y_test` | Separation stratifiee avant tout apprentissage statistique de pretraitement |
| 6. Pipeline de pretraitement | `ColumnTransformer` dans `Pipeline` | Imputation, traitement IQR, normalisation, encodage appris sur train puis appliques au test |
| 7. Modelisation | `X_pre`, `X_post`, `y_pre`, `y_post` | Separation des variables selon les usages pre-voyage et post-voyage, sur le meme dataset nettoye `df_model` |
| 8. Tracabilite | Notebook + MLflow | Documentation des choix, metriques et limites |

#### Conclusion

La preparation des donnees est couverte dans le perimetre du projet : le dataset est nettoye, structure, controle et prepare en fonction des deux cas d'usage identifies. Les operations statistiques sensibles sont integrees au pipeline afin d'eviter la fuite de donnees entre train et test. La validation effective par des parties prenantes externes reste une action a realiser hors notebook si le projet passe en contexte operationnel.


## 9. Fonctions de modelisation


In [ ]:

class IQRMedianOutlierReplacer(BaseEstimator, TransformerMixin):
    """Remplace les outliers IQR par la mediane apprise sur le train."""

    def __init__(self, factor: float = 1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=float)
        self.q1_ = np.nanquantile(X_array, 0.25, axis=0)
        self.q3_ = np.nanquantile(X_array, 0.75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        self.lower_bounds_ = self.q1_ - self.factor * self.iqr_
        self.upper_bounds_ = self.q3_ + self.factor * self.iqr_
        self.medians_ = np.nanmedian(X_array, axis=0)
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float).copy()
        outlier_mask = (
            (X_array < self.lower_bounds_)
            | (X_array > self.upper_bounds_)
        )
        return np.where(outlier_mask, self.medians_, X_array)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray([f"x{i}" for i in range(len(self.medians_))], dtype=object)
        return np.asarray(input_features, dtype=object)


def detecter_variables_binaires(X: pd.DataFrame, numeric_features: list[str]) -> list[str]:
    binary_features = []
    for column in numeric_features:
        values = pd.Series(X[column].dropna().unique())
        if values.empty:
            continue
        try:
            unique_values = set(values.astype(float).tolist())
        except (TypeError, ValueError):
            continue
        if unique_values.issubset({0.0, 1.0}):
            binary_features.append(column)
    return binary_features


def construire_preprocesseur(X: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

    binary_numeric_features = detecter_variables_binaires(X, numeric_features)
    continuous_numeric_features = [
        column for column in numeric_features
        if column not in binary_numeric_features
    ]

    continuous_numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("outliers_iqr", IQRMedianOutlierReplacer()),
        ("scaler", StandardScaler()),
    ])

    binary_numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = []
    if continuous_numeric_features:
        transformers.append(("num_cont", continuous_numeric_transformer, continuous_numeric_features))
    if binary_numeric_features:
        transformers.append(("num_bin", binary_numeric_transformer, binary_numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_transformer, categorical_features))

    preprocess = ColumnTransformer(transformers=transformers)

    return preprocess, numeric_features, categorical_features


def evaluer_classification(modeles: dict, X_train, X_test, y_train, y_test, preprocess) -> tuple[pd.DataFrame, dict]:
    rows = []
    fitted = {}

    for model_name, model in modeles.items():
        pipeline = Pipeline(steps=[
            ("preprocess", preprocess),
            ("model", model),
        ])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        rows.append({
            "modele": model_name,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "macro_f1": f1_score(y_test, predictions, average="macro"),
        })
        fitted[model_name] = pipeline

    results = (
        pd.DataFrame(rows)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )

    return results, fitted


def afficher_distribution(y: pd.Series, label: str) -> None:
    print(label)
    display(
        y.value_counts()
        .sort_index()
        .rename_axis("classe")
        .reset_index(name="nombre")
        .assign(pourcentage=lambda data: (data["nombre"] / data["nombre"].sum() * 100).round(2))
    )



## 10. Mod?le pr?-voyage

Objectif : pr?dire la satisfaction avant le d?part ? partir des informations disponibles au moment de la planification.

La cible est d?sormais regroup?e en 3 classes, comme pour le mod?le post-voyage :

- `0 = insatisfait` pour les notes 1 et 2 ;
- `1 = neutre` pour la note 3 ;
- `2 = satisfait` pour les notes 4 et 5.

R?gle cl? : les variables post-voyage sont exclues pour ?viter la fuite de donn?es.


In [ ]:
post_trip_columns = [
    "imprevus",
    "reorganisation_necessaire",
    "respect_budget",
    "retour_client",
    "imprevu_present",
    "imprevu_transport",
    "imprevu_meteo",
    "budget_non_respecte",
    "budget_tendu",
    "gravite_imprevu",
    "annulation_et_reorganisation",
    "retard_et_budget_non_respecte",
    "imprevu_transport_et_sejour_court",
    "budget_tendu_et_hebergement_luxe",
]

technical_columns = ["trip_id"]
excluded_pre_voyage = [TARGET_COLUMN, *technical_columns, *post_trip_columns]
feature_columns_pre = [
    column for column in df_model.columns
    if column not in excluded_pre_voyage
]

X_pre = df_model[feature_columns_pre].copy()
y_pre = df_model[TARGET_COLUMN].apply(regrouper_satisfaction_3_classes).astype(int).copy()

preprocess_pre, numeric_pre, categorical_pre = construire_preprocesseur(X_pre)

resume_pre = pd.DataFrame({
    "famille": ["numeriques", "categorielles", "total", "exclues"],
    "nombre": [len(numeric_pre), len(categorical_pre), X_pre.shape[1], len(excluded_pre_voyage)],
    "colonnes": [numeric_pre, categorical_pre, feature_columns_pre, excluded_pre_voyage],
})

display(resume_pre)
afficher_distribution(y_pre, "Distribution cible pr?-voyage ? 3 classes")


### 10.1 Corrélations pré-voyage


In [ ]:
X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(
    X_pre,
    y_pre,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_pre,
)

correlation_df_pre = X_train_pre[numeric_pre].copy()
correlation_df_pre["satisfaction_3_classes"] = y_train_pre.values

spearman_pre = (
    correlation_df_pre.corr(method="spearman")["satisfaction_3_classes"]
    .drop("satisfaction_3_classes")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

display(spearman_pre.to_frame("correlation_spearman_satisfaction_3_classes").round(4).head(15))


### 10.2 Comparaison des modèles pré-voyage


In [ ]:
modeles_pre = {
    "Dummy_majority_pre": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression_pre": LogisticRegression(max_iter=500, class_weight="balanced"),
    "RandomForest_pre": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=1,
    ),
    "ExtraTrees_pre": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results_pre, fitted_pre = evaluer_classification(
    modeles_pre,
    X_train_pre,
    X_test_pre,
    y_train_pre,
    y_test_pre,
    preprocess_pre,
)

display(results_pre.round(4))
best_pre_model_name = results_pre.iloc[0]["modele"]
print("Meilleur modèle pré-voyage :", best_pre_model_name)


### 10.3 Interpr?tation pr?-voyage

Le mod?le pr?-voyage est m?thodologiquement conforme au cas d'usage de planification, car il ne d?pend que de variables connues avant le s?jour.

La cible est regroup?e en 3 classes pour ?tre comparable au mod?le post-voyage. Ce regroupement rend le probl?me moins dispers? que la pr?diction directe des 5 notes de satisfaction.

Les performances restent toutefois limit?es. Cette limite est coh?rente avec les analyses pr?c?dentes : les caract?ristiques disponibles avant d?part contiennent peu de signal pour expliquer pr?cis?ment la satisfaction finale.

Le nettoyage strict des incoh?rences m?tier am?liore la qualit? du dataset, mais ne transforme pas le probl?me pr?-voyage en probl?me fortement pr?dictif. Il sert surtout ? ?viter d'entra?ner le mod?le sur des historiques contradictoires.



### 10.4 Exp?rience SMOTE pr?-voyage non retenue

Une exp?rience SMOTE a ?t? test?e sur la partie pr?-voyage afin de v?rifier si le d?s?quilibre des classes de `satisfaction_client` expliquait les faibles performances du mod?le.

SMOTE consiste ? cr?er artificiellement de nouveaux exemples pour les classes minoritaires. La m?thode a ?t? appliqu?e uniquement sur le jeu d'entra?nement, dans un pipeline `imblearn`, afin de ne pas modifier le jeu de test et d'?viter toute fuite de donn?es.

#### R?sultats observ?s apr?s passage du pr?-voyage en 3 classes

| Exp?rience | Meilleur mod?le | Accuracy | Balanced accuracy | Macro F1 |
| --- | --- | ---: | ---: | ---: |
| Sans SMOTE | `ExtraTrees_pre` | 0.3899 | 0.3698 | 0.3687 |
| Avec SMOTE | `RandomForest_pre_SMOTE` | 0.4128 | 0.3585 | 0.3523 |

Le gain obtenu sur le `macro_f1` est de `-0.0164`, donc l'augmentation artificielle des donn?es n'am?liore pas la performance globale du mod?le pr?-voyage.

#### Interpr?tation

Le regroupement en 3 classes am?liore la lisibilit? du probl?me pr?-voyage par rapport aux 5 notes originales, mais le signal disponible avant d?part reste limit?.

Le test SMOTE montre que le faible score du mod?le pr?-voyage ne vient pas principalement du d?s?quilibre des classes. Le probl?me vient plut?t du faible pouvoir explicatif des variables disponibles avant le d?part.

#### D?cision

SMOTE n'est pas retenu dans le pipeline final pr?-voyage. L'exp?rience est conserv?e uniquement comme ?l?ment de justification m?thodologique : elle montre qu'une m?thode d'augmentation de donn?es a ?t? test?e, mais qu'elle n'apporte pas d'am?lioration robuste dans ce contexte.



### Exp?rience SMOTE extr?me pr?-voyage : 20 000 lignes ajout?es

La m?me exp?rience extr?me est test?e sur le mod?le pr?-voyage afin de v?rifier si une forte augmentation artificielle peut compenser le faible signal disponible avant le d?part.

Comme pr?c?demment, l'objectif est d'ajouter exactement `20 000` lignes synth?tiques au jeu d'entra?nement, sans modifier le jeu de test.


In [ ]:

# Objectif : ajouter exactement 20 000 lignes synth?tiques sur le train pr?-voyage.
lignes_a_ajouter_smote_20000_pre = 20_000
lignes_finales_smote_20000_pre = len(y_train_pre) + lignes_a_ajouter_smote_20000_pre
classes_smote_20000_pre = sorted(pd.Series(y_train_pre).unique())
base_par_classe_smote_20000_pre = lignes_finales_smote_20000_pre // len(classes_smote_20000_pre)
reste_smote_20000_pre = lignes_finales_smote_20000_pre % len(classes_smote_20000_pre)

sampling_strategy_smote_20000_pre = {
    classe: base_par_classe_smote_20000_pre + (1 if index < reste_smote_20000_pre else 0)
    for index, classe in enumerate(classes_smote_20000_pre)
}

distribution_train_pre = pd.Series(y_train_pre).value_counts().sort_index()
volume_smote_20000_pre = pd.DataFrame([
    {
        "classe": classe,
        "lignes_avant_smote": int(distribution_train_pre.loc[classe]),
        "lignes_apres_smote_20000": int(sampling_strategy_smote_20000_pre[classe]),
        "lignes_ajoutees": int(sampling_strategy_smote_20000_pre[classe] - distribution_train_pre.loc[classe]),
    }
    for classe in classes_smote_20000_pre
])

modeles_smote_20000_pre = {
    "LogisticRegression_pre_SMOTE_20000": LogisticRegression(max_iter=500),
    "RandomForest_pre_SMOTE_20000": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "ExtraTrees_pre_SMOTE_20000": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

resultats_smote_20000_pre = []
for model_name, model in modeles_smote_20000_pre.items():
    pipeline = ImbPipeline(steps=[
        ("preprocess", preprocess_pre),
        ("smote", SMOTE(
            sampling_strategy=sampling_strategy_smote_20000_pre,
            random_state=RANDOM_STATE,
            k_neighbors=5,
        )),
        ("model", model),
    ])
    pipeline.fit(X_train_pre, y_train_pre)
    predictions = pipeline.predict(X_test_pre)

    resultats_smote_20000_pre.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test_pre, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test_pre, predictions),
        "macro_f1": f1_score(y_test_pre, predictions, average="macro"),
    })

resultats_smote_20000_pre = (
    pd.DataFrame(resultats_smote_20000_pre)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(volume_smote_20000_pre)
display(resultats_smote_20000_pre.round(4))



#### Interpr?tation SMOTE pr?-voyage +20 000 lignes

R?sultat observ? sur le split test :

| Mod?le | Accuracy | Balanced accuracy | Macro F1 |
| --- | ---: | ---: | ---: |
| `LogisticRegression_pre_SMOTE_20000` | 0.3486 | 0.3518 | 0.3421 |
| `RandomForest_pre_SMOTE_20000` | 0.3761 | 0.3332 | 0.3280 |
| `ExtraTrees_pre_SMOTE_20000` | 0.3440 | 0.3269 | 0.3266 |

Cette augmentation massive d?grade le meilleur mod?le pr?-voyage sans SMOTE (`ExtraTrees_pre`, `macro_f1 = 0.3687`).

Conclusion : le probl?me pr?-voyage ne vient pas principalement du manque de volume ou du d?s?quilibre des classes. Les variables disponibles avant d?part contiennent peu de signal pr?dictif ; ajouter 20 000 lignes synth?tiques ajoute surtout du bruit. SMOTE +20 000 lignes n'est pas retenu pour le pr?-voyage.



## 11. Mod?le post-voyage

Objectif : expliquer ou pr?dire la satisfaction apr?s le s?jour ? partir des ?v?nements op?rationnels observ?s.

Le mod?le post-voyage utilise le m?me dataset nettoy? `df_model` que le mod?le pr?-voyage. Les suppressions d'incoh?rences m?tier sont donc appliqu?es aux deux objectifs avant toute s?paration train/test.

Les variables `imprevus`, `reorganisation_necessaire` et `respect_budget` sont incluses car elles sont connues apr?s le s?jour et coh?rentes avec un objectif d'analyse qualit? post-voyage.

Le texte libre `retour_client` reste exclu du mod?le principal, car il refl?te directement le ressenti client et peut cr?er une fuite tr?s forte.


In [ ]:
def satisfaction_to_3_classes(value: int) -> int:
    if value <= 2:
        return 0
    if value == 3:
        return 1
    return 2


excluded_post_voyage = [
    "trip_id",
    TARGET_COLUMN,
    "retour_client",
    *features_supprimees_modelisation,
]

feature_columns_post = [
    column for column in df_model.columns
    if column not in excluded_post_voyage
]

X_post = df_model[feature_columns_post].copy()
y_post = df_model[TARGET_COLUMN].apply(satisfaction_to_3_classes).astype(int)

preprocess_post, numeric_post, categorical_post = construire_preprocesseur(X_post)

resume_post = pd.DataFrame({
    "famille": ["numeriques", "categorielles", "total", "exclues"],
    "nombre": [len(numeric_post), len(categorical_post), X_post.shape[1], len(excluded_post_voyage)],
    "colonnes": [numeric_post, categorical_post, feature_columns_post, excluded_post_voyage],
})

display(resume_post)
afficher_distribution(y_post, "Distribution cible post-voyage 3 classes")
print("Variables post-voyage explicatives incluses :")
print([column for column in features_post_voyage_explicatives if column in X_post.columns])

### 11.1 Équilibre des classes du modèle post-voyage

Avant l'entraînement, on vérifie la distribution de la cible `satisfaction_client` regroupée en 3 classes. Cette étape permet de savoir si le modèle risque de favoriser une classe majoritaire et justifie l'utilisation de métriques comme `macro_f1` et `balanced_accuracy`.


In [ ]:
X_train_post, X_test_post, y_train_post, y_test_post = train_test_split(
    X_post,
    y_post,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_post,
)

labels_satisfaction_3_classes = {
    0: "insatisfait_1_2",
    1: "neutre_3",
    2: "satisfait_4_5",
}


def calculer_equilibre_classes(y: pd.Series, jeu: str) -> pd.DataFrame:
    distribution = (
        y.value_counts()
        .sort_index()
        .rename_axis("classe")
        .reset_index(name="nombre")
    )
    distribution["jeu"] = jeu
    distribution["libelle"] = distribution["classe"].map(labels_satisfaction_3_classes)
    distribution["pourcentage"] = (
        distribution["nombre"] / distribution["nombre"].sum() * 100
    ).round(2)
    return distribution[["jeu", "classe", "libelle", "nombre", "pourcentage"]]


equilibre_classes_post = pd.concat(
    [
        calculer_equilibre_classes(y_post, "global"),
        calculer_equilibre_classes(y_train_post, "train"),
        calculer_equilibre_classes(y_test_post, "test"),
    ],
    ignore_index=True,
)

display(equilibre_classes_post)

classe_majoritaire = equilibre_classes_post[equilibre_classes_post["jeu"] == "global"].sort_values(
    "pourcentage",
    ascending=False,
).iloc[0]

print(
    "Classe majoritaire globale : "
    f"{classe_majoritaire['libelle']} "
    f"({classe_majoritaire['pourcentage']} %)"
)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
equilibre_global = equilibre_classes_post[equilibre_classes_post["jeu"] == "global"]
plt.bar(
    equilibre_global["libelle"],
    equilibre_global["pourcentage"],
    color=["#d95f02", "#7570b3", "#1b9e77"],
)
plt.title("Équilibre des classes - cible post-voyage")
plt.ylabel("Pourcentage des lignes (%)")
plt.xlabel("Classe de satisfaction")
plt.ylim(0, max(equilibre_global["pourcentage"]) + 10)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 11.2 Comparaison des modeles post-voyage 3 classes

La comparaison ne retient pas uniquement le meilleur score test. Le diagnostic precedent a montre qu'un modele tres flexible peut obtenir un score test legerement superieur tout en sur-apprenant fortement le jeu d'entrainement.

Le modele retenu dans cette section est donc un `RandomForestClassifier` regularise. Il sacrifie une petite partie du score test par rapport au `ExtraTreesClassifier` initial, mais reduit fortement l'overfitting et generalise mieux.


In [ ]:
modeles_post = {
    "Dummy_majority_3_classes": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression_3_classes": LogisticRegression(max_iter=500, class_weight="balanced"),
    "RandomForest_regularise_3_classes": RandomForestClassifier(
        n_estimators=120,
        max_depth=2,
        min_samples_leaf=30,
        min_samples_split=60,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results_post, fitted_post = evaluer_classification(
    modeles_post,
    X_train_post,
    X_test_post,
    y_train_post,
    y_test_post,
    preprocess_post,
)

display(results_post.round(4))
best_post_model_name = results_post.iloc[0]["modele"]
print("Meilleur mod?le post-voyage simple selon le macro_f1 :", best_post_model_name)


### 11.3 Optimisation Optuna anti-overfitting

Optuna est utilis? ici comme une exp?rience contr?l?e d'optimisation des hyperparam?tres du `RandomForestClassifier`.

La recherche respecte trois r?gles m?thodologiques :

- le jeu de test n'est pas utilis? pendant l'optimisation ;
- les essais sont ?valu?s par validation crois?e uniquement sur le jeu d'entra?nement ;
- l'objectif p?nalise les mod?les trop complexes ou pr?sentant un ?cart train/validation trop ?lev?.

Cette ?tape sert donc ? v?rifier s'il existe une meilleure configuration que le mod?le r?gularis? manuel, sans r?introduire de surapprentissage.


In [ ]:
# Optimisation Optuna contr?l?e du RandomForest.
# Le test set n'est pas utilis? dans la fonction objectif.
# L'objectif optimise le macro_f1 de validation crois?e tout en p?nalisant :
# - l'?cart train/validation au-del? de 0.05 ;
# - la complexit? du mod?le pour tenir compte de la RAM et de l'?co-conception.

import math

optuna.logging.set_verbosity(optuna.logging.WARNING)

search_space_rf = {
    "n_estimators": [80, 120],
    "max_depth": [2, 3],
    "min_samples_leaf": [30, 40, 50],
    "min_samples_split": [60, 100],
    "max_features": ["sqrt", None],
}


def objectif_random_forest_optuna(trial):
    params = {
        key: trial.suggest_categorical(key, values)
        for key, values in search_space_rf.items()
    }
    params.update({
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
    })

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    train_macro_f1_scores = []
    validation_macro_f1_scores = []

    for train_index, validation_index in cv.split(X_train_post, y_train_post):
        X_train_fold = X_train_post.iloc[train_index]
        X_validation_fold = X_train_post.iloc[validation_index]
        y_train_fold = y_train_post.iloc[train_index]
        y_validation_fold = y_train_post.iloc[validation_index]

        pipeline = Pipeline(steps=[
            ("preprocess", preprocess_post),
            ("model", RandomForestClassifier(**params)),
        ])
        pipeline.fit(X_train_fold, y_train_fold)

        train_macro_f1_scores.append(
            f1_score(y_train_fold, pipeline.predict(X_train_fold), average="macro")
        )
        validation_macro_f1_scores.append(
            f1_score(y_validation_fold, pipeline.predict(X_validation_fold), average="macro")
        )

    train_macro_f1_mean = float(np.mean(train_macro_f1_scores))
    validation_macro_f1_mean = float(np.mean(validation_macro_f1_scores))
    gap_macro_f1 = train_macro_f1_mean - validation_macro_f1_mean

    overfit_penalty = max(gap_macro_f1 - 0.05, 0) * 2.0
    complexity_penalty = (
        (params["n_estimators"] - 80) * 0.0003
        + (params["max_depth"] - 2) * 0.006
    )

    objective_score = validation_macro_f1_mean - overfit_penalty - complexity_penalty

    trial.set_user_attr("train_macro_f1_mean", train_macro_f1_mean)
    trial.set_user_attr("cv_macro_f1_mean", validation_macro_f1_mean)
    trial.set_user_attr("gap_macro_f1", gap_macro_f1)
    trial.set_user_attr("overfit_penalty", overfit_penalty)
    trial.set_user_attr("complexity_penalty", complexity_penalty)

    return objective_score


study_rf_optuna = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(search_space_rf, seed=RANDOM_STATE),
    study_name="random_forest_regularise_anti_overfitting",
)
study_rf_optuna.optimize(
    objectif_random_forest_optuna,
    n_trials=math.prod(len(values) for values in search_space_rf.values()),
    show_progress_bar=False,
)

best_trial_rf_optuna = study_rf_optuna.best_trial
best_params_rf_optuna = dict(best_trial_rf_optuna.params)
best_params_rf_optuna.update({
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
})

modeles_post["RandomForest_Optuna_regularise_3_classes"] = RandomForestClassifier(
    **best_params_rf_optuna
)

optuna_pipeline_post = Pipeline(steps=[
    ("preprocess", preprocess_post),
    ("model", RandomForestClassifier(**best_params_rf_optuna)),
])
optuna_pipeline_post.fit(X_train_post, y_train_post)
fitted_post["RandomForest_Optuna_regularise_3_classes"] = optuna_pipeline_post

optuna_train_predictions = optuna_pipeline_post.predict(X_train_post)
optuna_test_predictions = optuna_pipeline_post.predict(X_test_post)

optuna_metrics_post = pd.DataFrame([
    {
        "jeu": "train",
        "accuracy": accuracy_score(y_train_post, optuna_train_predictions),
        "balanced_accuracy": balanced_accuracy_score(y_train_post, optuna_train_predictions),
        "macro_f1": f1_score(y_train_post, optuna_train_predictions, average="macro"),
    },
    {
        "jeu": "test",
        "accuracy": accuracy_score(y_test_post, optuna_test_predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test_post, optuna_test_predictions),
        "macro_f1": f1_score(y_test_post, optuna_test_predictions, average="macro"),
    },
])

optuna_gap_train_test_macro_f1 = (
    optuna_metrics_post.loc[optuna_metrics_post["jeu"] == "train", "macro_f1"].iloc[0]
    - optuna_metrics_post.loc[optuna_metrics_post["jeu"] == "test", "macro_f1"].iloc[0]
)

optuna_result_row = {
    "modele": "RandomForest_Optuna_regularise_3_classes",
    "accuracy": optuna_metrics_post.loc[optuna_metrics_post["jeu"] == "test", "accuracy"].iloc[0],
    "balanced_accuracy": optuna_metrics_post.loc[optuna_metrics_post["jeu"] == "test", "balanced_accuracy"].iloc[0],
    "macro_f1": optuna_metrics_post.loc[optuna_metrics_post["jeu"] == "test", "macro_f1"].iloc[0],
}

comparaison_optuna_post = pd.concat(
    [
        results_post,
        pd.DataFrame([optuna_result_row]),
    ],
    ignore_index=True,
).sort_values("macro_f1", ascending=False).reset_index(drop=True)

resume_optuna_post = pd.DataFrame([
    {
        "element": "best_params",
        "valeur": best_params_rf_optuna,
    },
    {
        "element": "objective_score",
        "valeur": round(best_trial_rf_optuna.value, 4),
    },
    {
        "element": "cv_macro_f1_train_interne",
        "valeur": round(best_trial_rf_optuna.user_attrs["cv_macro_f1_mean"], 4),
    },
    {
        "element": "gap_macro_f1_train_validation_interne",
        "valeur": round(best_trial_rf_optuna.user_attrs["gap_macro_f1"], 4),
    },
    {
        "element": "gap_macro_f1_train_test",
        "valeur": round(optuna_gap_train_test_macro_f1, 4),
    },
])

optuna_trials_rf = (
    study_rf_optuna.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
    .sort_values("value", ascending=False)
    .head(10)
)

print("Meilleurs hyperparam?tres Optuna :")
display(resume_optuna_post)

print("Top 10 des essais Optuna :")
display(optuna_trials_rf.round(4))

print("Scores du mod?le Optuna :")
display(optuna_metrics_post.round(4))

print("Comparaison avec les mod?les post-voyage existants :")
display(comparaison_optuna_post.round(4))

baseline_macro_f1 = results_post.loc[
    results_post["modele"] == "RandomForest_regularise_3_classes",
    "macro_f1",
].iloc[0]
optuna_macro_f1 = optuna_result_row["macro_f1"]

if optuna_macro_f1 > baseline_macro_f1 and optuna_gap_train_test_macro_f1 <= 0.07:
    print("Conclusion : Optuna am?liore le macro_f1 du RandomForest sans signe fort d'overfitting. Candidat ? comparer aux autres mod?les.")
else:
    print(
        "Conclusion : Optuna ne produit pas une am?lioration suffisamment robuste. "
        "Le RandomForest r?gularis? manuel reste la r?f?rence RandomForest ; la d?cision finale reste bas?e sur results_post/comparison_post."
    )



### 11.4 Synthese des ameliorations et de l'overfitting

Cette section documente les essais r?alis?s pour am?liorer le mod?le post-voyage sans cr?er de fuite de donn?es.

#### Point de d?part

Un mod?le d'arbres trop flexible peut obtenir un score tr?s ?lev? sur le train mais un score beaucoup plus faible sur le test. Ce comportement correspond ? de l'overfitting : le mod?le m?morise les cas d'entra?nement au lieu de g?n?raliser.

#### Correction m?thodologique anti-fuite

L'imputation, la standardisation, l'encodage et le traitement des outliers IQR ont ?t? d?plac?s dans le pipeline `scikit-learn`. Les param?tres sont donc appris uniquement sur le train, puis appliqu?s au test.

Cette correction rend l'?valuation plus rigoureuse et modifie les scores observ?s par rapport aux versions pr?c?dentes du notebook.

#### Optimisation manuelle anti-overfitting

Une version `RandomForestClassifier` fortement r?gularis?e a ?t? test?e avec :

| Hyperparam?tre | Valeur test?e | Effet recherch? |
| --- | ---: | --- |
| `n_estimators` | 120 | Garder un ensemble stable d'arbres |
| `max_depth` | 2 | Limiter la complexit? des arbres |
| `min_samples_leaf` | 30 | Eviter les feuilles trop sp?cifiques |
| `min_samples_split` | 60 | Eviter les divisions sur peu d'exemples |
| `max_features` | `sqrt` | Introduire de la diversit? entre les arbres |
| `class_weight` | `balanced` | Tenir compte du d?s?quilibre des classes |

Apr?s correction du pipeline, ce mod?le obtient environ `macro_f1 = 0.4015` sur le split test. Il reste plus stable qu'un mod?le tr?s flexible, mais il ne d?passe pas la r?gression logistique simple sur le crit?re principal.

#### Optimisation automatique Optuna

Optuna a ensuite ?t? utilis? pour v?rifier si une recherche automatique d'hyperparam?tres pouvait am?liorer le `RandomForestClassifier` r?gularis?.

Meilleurs hyperparam?tres trouv?s :

| Hyperparam?tre | Valeur Optuna |
| --- | ---: |
| `n_estimators` | 80 |
| `max_depth` | 2 |
| `min_samples_leaf` | 50 |
| `min_samples_split` | 60 |
| `max_features` | `sqrt` |
| `class_weight` | `balanced` |

R?sultats Optuna recalcul?s avec le pipeline sans fuite :

| Jeu / validation | Accuracy | Balanced accuracy | Macro F1 |
| --- | ---: | ---: | ---: |
| Train | 0.5103 | 0.4279 | 0.3822 |
| Test | 0.5367 | 0.4538 | 0.3983 |
| Validation crois?e moyenne | 0.5027 | 0.4251 | 0.3854 |

L'?cart `macro_f1 train-test` est de `-0.0160`, donc Optuna ne montre pas de surapprentissage fort. En revanche, le `macro_f1 test = 0.3983` reste inf?rieur au mod?le post-voyage simple `LogisticRegression_3_classes`, qui obtient `macro_f1 = 0.4392`.

#### D?cision m?thodologique

Optuna am?liore l?g?rement la configuration RandomForest r?gularis?e sur certains indicateurs, mais ne d?passe pas le meilleur mod?le simple. L'optimisation automatique est donc document?e comme exp?rience de contr?le, mais elle n'est pas retenue comme mod?le final.

La d?cision finale ne d?pend pas uniquement de l'accuracy. Le crit?re principal reste le `macro_f1`, car il tient mieux compte des classes minoritaires. Les r?sultats doivent ?tre relus apr?s chaque modification de nettoyage ou de feature engineering.



### 11.5 Tests compl?mentaires : SMOTE, RandomForest optimis? et XGBoost

Cette section teste des am?liorations suppl?mentaires sur le mod?le post-voyage, apr?s le nettoyage strict des incoh?rences m?tier et apr?s d?placement de l'imputation / traitement IQR dans le pipeline.

Les exp?riences restent s?par?es du mod?le simple de r?f?rence afin de comparer objectivement leur apport :

- `SMOTE` : r??quilibrage artificiel des classes, appliqu? uniquement sur le jeu d'entra?nement ;
- `RandomForest_3_classes_optimise_500` : for?t plus complexe avec davantage d'arbres ;
- `XGBoost_3_classes_optimise` : mod?le de boosting tabulaire plus puissant mais plus co?teux.

#### R?sultat observ? sur le split test

Lors du dernier test cibl?, le meilleur r?sultat compl?mentaire est obtenu par `RandomForest_3_classes_optimise_500` avec environ :

| Mod?le | Accuracy | Balanced accuracy | Macro F1 |
| --- | ---: | ---: | ---: |
| `RandomForest_3_classes_optimise_500` | 0.5229 | 0.4726 | 0.4702 |
| `RandomForest_3_classes_SMOTE` | 0.5321 | 0.4613 | 0.4486 |
| `LogisticRegression_3_classes` | 0.4587 | 0.4438 | 0.4392 |
| `ExtraTrees_3_classes_SMOTE` | 0.5000 | 0.4382 | 0.4287 |
| `LogisticRegression_3_classes_SMOTE` | 0.4495 | 0.4278 | 0.4236 |
| `RandomForest_regularise_3_classes` | 0.5413 | 0.4571 | 0.4015 |
| `XGBoost_3_classes_optimise` | 0.4312 | 0.3801 | 0.3730 |
| `Dummy_majority_3_classes` | 0.4633 | 0.3333 | 0.2111 |

#### Interpr?tation

Le split test montre une am?lioration avec un RandomForest plus complexe et avec certains pipelines SMOTE. Cependant, ces r?sultats doivent ?tre lus avec prudence : un seul d?coupage train/test peut ?tre favorable ? un mod?le.

La validation crois?e ci-dessous est donc utilis?e pour v?rifier si l'am?lioration SMOTE est robuste. La d?cision finale ne doit pas se baser uniquement sur le meilleur score ponctuel du split test.


In [ ]:
tests_complementaires = []

# SMOTE.
smote_modeles = {
    "LogisticRegression_3_classes_SMOTE": LogisticRegression(max_iter=500),
    "RandomForest_3_classes_SMOTE": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "ExtraTrees_3_classes_SMOTE": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

for model_name, model in smote_modeles.items():
    pipeline = ImbPipeline(steps=[
        ("preprocess", preprocess_post),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", model),
    ])
    pipeline.fit(X_train_post, y_train_post)
    predictions = pipeline.predict(X_test_post)
    tests_complementaires.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test_post, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test_post, predictions),
        "macro_f1": f1_score(y_test_post, predictions, average="macro"),
    })

# RandomForest optimisé manuellement.
rf_optim_pipeline = Pipeline(steps=[
    ("preprocess", preprocess_post),
    ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=5,
        min_samples_split=10,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=1,
    )),
])
rf_optim_pipeline.fit(X_train_post, y_train_post)
rf_optim_predictions = rf_optim_pipeline.predict(X_test_post)
tests_complementaires.append({
    "modele": "RandomForest_3_classes_optimise_500",
    "accuracy": accuracy_score(y_test_post, rf_optim_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test_post, rf_optim_predictions),
    "macro_f1": f1_score(y_test_post, rf_optim_predictions, average="macro"),
})

# XGBoost optimisé manuellement.
xgboost_pipeline = Pipeline(steps=[
    ("preprocess", preprocess_post),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])
xgboost_pipeline.fit(X_train_post, y_train_post)
xgboost_predictions = xgboost_pipeline.predict(X_test_post)
tests_complementaires.append({
    "modele": "XGBoost_3_classes_optimise",
    "accuracy": accuracy_score(y_test_post, xgboost_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test_post, xgboost_predictions),
    "macro_f1": f1_score(y_test_post, xgboost_predictions, average="macro"),
})

results_post_complementaires = pd.DataFrame(tests_complementaires)
comparison_post = (
    pd.concat([results_post, results_post_complementaires], ignore_index=True)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(comparison_post.round(4))


#### Validation crois?e des exp?riences SMOTE

Le score obtenu sur un seul jeu de test peut ?tre favorable par hasard. Pour valider SMOTE plus rigoureusement, on refait donc une validation crois?e ? 3 plis.

La r?gle m?thodologique est importante : SMOTE doit ?tre plac? dans un `ImbPipeline`, apr?s le pr?traitement et avant le mod?le. Ainsi, les exemples synth?tiques sont cr??s uniquement sur les folds d'entra?nement, jamais sur les folds de validation.


In [ ]:

modeles_cv_smote_post = {
    "LogisticRegression_sans_SMOTE": fitted_post["LogisticRegression_3_classes"],
    "RandomForest_regularise_sans_SMOTE": fitted_post["RandomForest_regularise_3_classes"],
    "LogisticRegression_avec_SMOTE": ImbPipeline(steps=[
        ("preprocess", preprocess_post),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", LogisticRegression(max_iter=500)),
    ]),
    "RandomForest_avec_SMOTE": ImbPipeline(steps=[
        ("preprocess", preprocess_post),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", RandomForestClassifier(
            n_estimators=120,
            max_depth=8,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
    "ExtraTrees_avec_SMOTE": ImbPipeline(steps=[
        ("preprocess", preprocess_post),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", ExtraTreesClassifier(
            n_estimators=120,
            max_depth=8,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
}

cv_smote_rows = []
for model_name, pipeline in modeles_cv_smote_post.items():
    cv_results = cross_validate(
        pipeline,
        X_post,
        y_post,
        cv=3,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro",
        },
        n_jobs=1,
    )

    cv_smote_rows.append({
        "modele": model_name,
        "accuracy_moyenne": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "balanced_accuracy_moyenne": cv_results["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": cv_results["test_balanced_accuracy"].std(),
        "macro_f1_moyenne": cv_results["test_macro_f1"].mean(),
        "macro_f1_std": cv_results["test_macro_f1"].std(),
    })

cv_smote_post = (
    pd.DataFrame(cv_smote_rows)
    .sort_values("macro_f1_moyenne", ascending=False)
    .reset_index(drop=True)
)

display(cv_smote_post.round(4))



#### R?sultats trac?s de la validation crois?e SMOTE

Les r?sultats obtenus lors de la validation crois?e ? 3 plis sont les suivants :

| Mod?le | Macro F1 moyen |
| --- | ---: |
| `ExtraTrees_avec_SMOTE` | 0.4101 |
| `RandomForest_regularise_sans_SMOTE` | 0.3901 |
| `RandomForest_avec_SMOTE` | 0.3870 |
| `LogisticRegression_avec_SMOTE` | 0.3846 |
| `LogisticRegression_sans_SMOTE` | 0.3758 |

Ces r?sultats montrent que SMOTE apporte un gain mod?r? uniquement avec `ExtraTrees`. Le gain n'est pas suffisamment net pour retenir automatiquement SMOTE comme mod?le final sans validation m?tier compl?mentaire.



#### Interpr?tation de la validation crois?e SMOTE

Sur le split test, les r?sultats SMOTE semblent int?ressants, notamment avec `RandomForest_3_classes_SMOTE`. En validation crois?e, le meilleur score moyen est obtenu par `ExtraTrees_avec_SMOTE`, autour de `macro_f1 = 0.4101`.

Le gain existe par rapport ? certaines versions sans SMOTE, mais il reste mod?r?. La validation crois?e ne confirme donc pas un gain suffisamment fort pour retenir automatiquement SMOTE comme mod?le final.

Conclusion : SMOTE reste une piste d'am?lioration post-voyage ? documenter, mais pas une d?cision d?finitive. Le mod?le final doit ?tre choisi en arbitrant performance, stabilit?, simplicit? et validation m?tier.



### Tra?abilit? du volume ajout? par SMOTE

SMOTE est appliqu? uniquement sur le jeu d'entra?nement. Le jeu de test n'est pas modifi?.

Comme les mod?les pr?-voyage et post-voyage utilisent d?sormais la m?me cible regroup?e en 3 classes, la distribution de `y_train` est identique pour les deux exp?riences. SMOTE r??quilibre chaque classe au niveau de la classe majoritaire.


In [ ]:

def calculer_volume_smote(y_train: pd.Series, experience: str) -> pd.DataFrame:
    distribution_avant = pd.Series(y_train).value_counts().sort_index()
    effectif_majoritaire = int(distribution_avant.max())
    distribution_apres = pd.Series(
        {classe: effectif_majoritaire for classe in distribution_avant.index}
    ).sort_index()

    return pd.DataFrame({
        "experience": experience,
        "classe": distribution_avant.index,
        "lignes_avant_smote": distribution_avant.values,
        "lignes_apres_smote": distribution_apres.values,
        "lignes_ajoutees": distribution_apres.values - distribution_avant.values,
    })

volume_smote_pre = calculer_volume_smote(y_train_pre, "pre_voyage")
volume_smote_post = calculer_volume_smote(y_train_post, "post_voyage")
volume_smote = pd.concat([volume_smote_pre, volume_smote_post], ignore_index=True)

resume_volume_smote = (
    volume_smote
    .groupby("experience", as_index=False)
    .agg(
        lignes_train_avant=("lignes_avant_smote", "sum"),
        lignes_train_apres=("lignes_apres_smote", "sum"),
        lignes_ajoutees=("lignes_ajoutees", "sum"),
    )
)

display(volume_smote)
display(resume_volume_smote)



### Exp?rience SMOTE extr?me : 20 000 lignes ajout?es

Une exp?rience volontairement extr?me est test?e pour v?rifier si une forte augmentation artificielle du train am?liore le mod?le post-voyage.

L'objectif est d'ajouter exactement `20 000` lignes synth?tiques au jeu d'entra?nement, sans modifier le jeu de test. Cette exp?rience n'est pas recommand?e par d?faut : elle sert uniquement ? v?rifier si le manque de volume explique les limites de performance.


In [ ]:

# Objectif : ajouter exactement 20 000 lignes synth?tiques sur le train post-voyage.
# Train initial : 870 lignes.
# Train apr?s SMOTE extr?me : 20 870 lignes.

lignes_a_ajouter_smote_20000 = 20_000
lignes_finales_smote_20000 = len(y_train_post) + lignes_a_ajouter_smote_20000
classes_smote_20000 = sorted(pd.Series(y_train_post).unique())
base_par_classe_smote_20000 = lignes_finales_smote_20000 // len(classes_smote_20000)
reste_smote_20000 = lignes_finales_smote_20000 % len(classes_smote_20000)

sampling_strategy_smote_20000 = {
    classe: base_par_classe_smote_20000 + (1 if index < reste_smote_20000 else 0)
    for index, classe in enumerate(classes_smote_20000)
}

distribution_train_post = pd.Series(y_train_post).value_counts().sort_index()
volume_smote_20000 = pd.DataFrame([
    {
        "classe": classe,
        "lignes_avant_smote": int(distribution_train_post.loc[classe]),
        "lignes_apres_smote_20000": int(sampling_strategy_smote_20000[classe]),
        "lignes_ajoutees": int(sampling_strategy_smote_20000[classe] - distribution_train_post.loc[classe]),
    }
    for classe in classes_smote_20000
])

modeles_smote_20000 = {
    "LogisticRegression_SMOTE_20000": LogisticRegression(max_iter=500),
    "RandomForest_SMOTE_20000": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "ExtraTrees_SMOTE_20000": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

resultats_smote_20000 = []
for model_name, model in modeles_smote_20000.items():
    pipeline = ImbPipeline(steps=[
        ("preprocess", preprocess_post),
        ("smote", SMOTE(
            sampling_strategy=sampling_strategy_smote_20000,
            random_state=RANDOM_STATE,
            k_neighbors=5,
        )),
        ("model", model),
    ])
    pipeline.fit(X_train_post, y_train_post)
    predictions = pipeline.predict(X_test_post)

    resultats_smote_20000.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test_post, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test_post, predictions),
        "macro_f1": f1_score(y_test_post, predictions, average="macro"),
    })

resultats_smote_20000 = (
    pd.DataFrame(resultats_smote_20000)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(volume_smote_20000)
display(resultats_smote_20000.round(4))



#### Interpr?tation SMOTE +20 000 lignes

R?sultat observ? sur le split test :

| Mod?le | Accuracy | Balanced accuracy | Macro F1 |
| --- | ---: | ---: | ---: |
| `RandomForest_SMOTE_20000` | 0.4954 | 0.4447 | 0.4389 |
| `LogisticRegression_SMOTE_20000` | 0.4495 | 0.4428 | 0.4369 |
| `ExtraTrees_SMOTE_20000` | 0.4725 | 0.4339 | 0.4348 |

Cette augmentation massive n'am?liore pas le meilleur score post-voyage simple (`LogisticRegression_3_classes`, `macro_f1 = 0.4392`) et reste inf?rieure au test compl?mentaire `RandomForest_3_classes_optimise_500` (`macro_f1 = 0.4702`).

Conclusion : ajouter beaucoup de lignes synth?tiques ne cr?e pas de signal m?tier suppl?mentaire. SMOTE +20 000 lignes n'est pas retenu.


### 11.6 Validation crois?e du mod?le post-voyage retenu

La validation croisée vérifie si le score du modèle retenu est stable sur plusieurs découpages du dataset.

Elle complète le simple découpage train/test et permet de vérifier que le résultat n'est pas seulement lié à un split favorable.


In [ ]:

# Validation crois?e du mod?le post-voyage simple retenu dans results_post.
# cross_validate clone le pipeline ; les pr?traitements restent donc appris uniquement sur les folds train.
pipeline_cv_post = fitted_post[best_post_model_name]

cv_results_post = cross_validate(
    pipeline_cv_post,
    X_post,
    y_post,
    cv=3,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro",
    },
    n_jobs=1,
)

cv_summary_post = pd.DataFrame({
    "metrique": ["accuracy", "balanced_accuracy", "macro_f1"],
    "moyenne": [
        cv_results_post["test_accuracy"].mean(),
        cv_results_post["test_balanced_accuracy"].mean(),
        cv_results_post["test_macro_f1"].mean(),
    ],
    "ecart_type": [
        cv_results_post["test_accuracy"].std(),
        cv_results_post["test_balanced_accuracy"].std(),
        cv_results_post["test_macro_f1"].std(),
    ],
})

print("Mod?le valid? par validation crois?e :", best_post_model_name)
display(cv_summary_post.round(4))



### 11.7 Diagnostic d'overfitting du modele retenu

Le diagnostic d'overfitting compare les performances du mod?le retenu sur le train, le test et la validation crois?e.

Principe d'interpr?tation :

- si le score train est tr?s sup?rieur au score test, le mod?le m?morise trop les donn?es d'entra?nement ;
- si le score test est proche du score de validation crois?e, le r?sultat est plus stable ;
- si les scores sont faibles sur train et test, le probl?me est plut?t un manque de signal ou un sous-apprentissage.

Apr?s le nettoyage strict, ce diagnostic est recalcul? automatiquement sur le mod?le s?lectionn? dans `results_post` selon le `macro_f1`.


In [ ]:
best_model_name = results_post.iloc[0]["modele"]
best_post_pipeline = fitted_post[best_model_name]

train_predictions_overfit = best_post_pipeline.predict(X_train_post)
test_predictions_overfit = best_post_pipeline.predict(X_test_post)

overfitting_diagnostic = pd.DataFrame([
    {
        "jeu": "train",
        "accuracy": accuracy_score(y_train_post, train_predictions_overfit),
        "balanced_accuracy": balanced_accuracy_score(y_train_post, train_predictions_overfit),
        "macro_f1": f1_score(y_train_post, train_predictions_overfit, average="macro"),
    },
    {
        "jeu": "test",
        "accuracy": accuracy_score(y_test_post, test_predictions_overfit),
        "balanced_accuracy": balanced_accuracy_score(y_test_post, test_predictions_overfit),
        "macro_f1": f1_score(y_test_post, test_predictions_overfit, average="macro"),
    },
    {
        "jeu": "validation_croisee_moyenne",
        "accuracy": cv_results_post["test_accuracy"].mean(),
        "balanced_accuracy": cv_results_post["test_balanced_accuracy"].mean(),
        "macro_f1": cv_results_post["test_macro_f1"].mean(),
    },
])

gap_train_test_macro_f1 = (
    overfitting_diagnostic.loc[overfitting_diagnostic["jeu"] == "train", "macro_f1"].iloc[0]
    - overfitting_diagnostic.loc[overfitting_diagnostic["jeu"] == "test", "macro_f1"].iloc[0]
)

display(overfitting_diagnostic.round(4))
print(f"Ecart macro_f1 train-test : {gap_train_test_macro_f1:.4f}")

if gap_train_test_macro_f1 > 0.15:
    print("Conclusion : risque d'overfitting eleve.")
elif gap_train_test_macro_f1 > 0.07:
    print("Conclusion : risque d'overfitting modere a surveiller.")
else:
    print("Conclusion : pas de signe fort d'overfitting sur ce diagnostic.")

# Visualisation explicite de l'ecart train/test sous forme de courbes.
# Les trois metriques sont affichees pour verifier si le train reste proche du test.
metrics_overfit = ["accuracy", "balanced_accuracy", "macro_f1"]
labels_overfit = ["Accuracy", "Balanced accuracy", "Macro F1"]

train_values_overfit = [
    overfitting_diagnostic.loc[
        overfitting_diagnostic["jeu"] == "train",
        metric,
    ].iloc[0]
    for metric in metrics_overfit
]
test_values_overfit = [
    overfitting_diagnostic.loc[
        overfitting_diagnostic["jeu"] == "test",
        metric,
    ].iloc[0]
    for metric in metrics_overfit
]
cv_values_overfit = [
    overfitting_diagnostic.loc[
        overfitting_diagnostic["jeu"] == "validation_croisee_moyenne",
        metric,
    ].iloc[0]
    for metric in metrics_overfit
]

x_positions = np.arange(len(metrics_overfit))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    x_positions,
    train_values_overfit,
    marker="o",
    linewidth=2.5,
    color="#4c78a8",
    label="Train",
)
ax.plot(
    x_positions,
    test_values_overfit,
    marker="o",
    linewidth=2.5,
    color="#f58518",
    label="Test",
)
ax.plot(
    x_positions,
    cv_values_overfit,
    marker="o",
    linestyle="--",
    linewidth=2,
    color="#54a24b",
    label="Validation croisee moyenne",
)

for x_position, train_value, test_value in zip(
    x_positions,
    train_values_overfit,
    test_values_overfit,
):
    ax.vlines(
        x=x_position,
        ymin=min(train_value, test_value),
        ymax=max(train_value, test_value),
        color="#d62728",
        linestyle=":",
        linewidth=2,
        alpha=0.8,
    )
    ax.text(
        x_position + 0.04,
        (train_value + test_value) / 2,
        f"ecart {abs(train_value - test_value):.3f}",
        color="#d62728",
        fontsize=9,
        va="center",
    )

ax.set_title(f"Courbes train/test - diagnostic d'overfitting\n{best_model_name}")
ax.set_xticks(x_positions)
ax.set_xticklabels(labels_overfit)
ax.set_ylabel("Score")
ax.set_ylim(0, max(0.65, max(train_values_overfit + test_values_overfit + cv_values_overfit) + 0.12))
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()


### 11.8 Diagnostic du modele post-voyage retenu


In [ ]:
# Le diagnostic final est realise sur le meilleur modele simple,
# coherent avec le pipeline industrialise.
best_model_name = results_post.iloc[0]["modele"]
best_post_pipeline = fitted_post[best_model_name]

best_predictions = best_post_pipeline.predict(X_test_post)

labels_3_classes = [0, 1, 2]
labels_readable = ["insatisfait_1_2", "neutre_3", "satisfait_4_5"]

confusion_post = pd.DataFrame(
    confusion_matrix(y_test_post, best_predictions, labels=labels_3_classes),
    index=[f"reel_{label}" for label in labels_readable],
    columns=[f"predit_{label}" for label in labels_readable],
)

report_post = pd.DataFrame(
    classification_report(
        y_test_post,
        best_predictions,
        labels=labels_3_classes,
        target_names=labels_readable,
        output_dict=True,
        zero_division=0,
    )
).transpose()

print("Modèle retenu pour diagnostic :", best_model_name)

### 11.9 Matrice de confusion du modele retenu

La matrice de confusion compare les classes réelles aux classes prédites.

- Les lignes correspondent aux classes réelles.
- Les colonnes correspondent aux classes prédites.
- La diagonale correspond aux bonnes prédictions.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Matrice de confusion — lignes = réel, colonnes = prédit")
display(confusion_post)

plt.figure(figsize=(9, 6))
sns.heatmap(
    confusion_post,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
)
plt.title(f"Matrice de confusion — {best_model_name}")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.tight_layout()
plt.show()

### 11.10 Rapport de classification du modele retenu


In [ ]:
display(report_post.round(4))

### 11.11 Importance des variables du modele retenu


In [ ]:
if hasattr(best_post_pipeline.named_steps["model"], "feature_importances_"):
    feature_names = best_post_pipeline.named_steps["preprocess"].get_feature_names_out()
    importances = best_post_pipeline.named_steps["model"].feature_importances_

    importance_df = (
        pd.DataFrame({
            "feature": feature_names,
            "importance": importances,
        })
        .sort_values("importance", ascending=False)
        .head(20)
    )
    display(importance_df.round(4))
else:
    print("Le modèle retenu ne fournit pas d'importance des variables directement exploitable.")

### 11.12 Tracabilite MLflow


In [ ]:
with mlflow.start_run(run_name=f"post_voyage_{best_model_name}"):
    mlflow.log_param("objectif", "post-voyage")
    mlflow.log_param("target", "satisfaction_client_3_classes")
    mlflow.log_param("modele", best_model_name)
    mlflow.log_param("nb_features", len(feature_columns_post))
    mlflow.log_param("features", ", ".join(feature_columns_post))

    best_row = results_post[results_post["modele"] == best_model_name].iloc[0]
    mlflow.log_metric("accuracy", best_row["accuracy"])
    mlflow.log_metric("balanced_accuracy", best_row["balanced_accuracy"])
    mlflow.log_metric("macro_f1", best_row["macro_f1"])

    mlflow.log_metric(
        "cv_macro_f1_mean",
        cv_summary_post.loc[cv_summary_post["metrique"] == "macro_f1", "moyenne"].iloc[0]
    )

    if "gap_train_test_macro_f1" in globals():
        mlflow.log_metric("gap_train_test_macro_f1", gap_train_test_macro_f1)

    mlflow.sklearn.log_model(best_post_pipeline, "model")

print("Run MLflow enregistre")



### 11.13 Choix du modele IA

Cette section formalise le choix du modele IA au regard des cas d'usage, des performances observees, des contraintes operationnelles et des contraintes d'eco-conception.

#### Demarche scientifique de selection

La selection du modele repose sur une demarche comparative :

1. definir clairement le cas d'usage : prediction / explication post-voyage de la satisfaction en 3 classes ;
2. nettoyer les incoherences metier avant toute separation train/test ;
3. separer les donnees en train/test avec stratification pour conserver l'equilibre des classes ;
4. comparer plusieurs familles de modeles avec les memes donnees et les memes metriques ;
5. comparer les modeles a une baseline naive `DummyClassifier` ;
6. verifier la stabilite par validation croisee ;
7. tester des ameliorations controlees : regularisation, SMOTE, XGBoost et Optuna ;
8. analyser les erreurs avec la matrice de confusion et le rapport de classification ;
9. verifier la capacite probabiliste du modele avec une analyse ROC multiclasses ;
10. documenter les limites et contraintes dans la Model Card.

#### Familles d'algorithmes considerees

| Famille | Modele teste | Interet | Contraintes / limites |
| --- | --- | --- | --- |
| Baseline naive | `DummyClassifier` | Point de comparaison minimal | Ne capture aucun signal metier |
| Modele lineaire | `LogisticRegression` | Simple, rapide, interpretable | Peut sous-performer si relations non lineaires |
| Ensemble bagging | `RandomForestClassifier` | Robuste, gere interactions non lineaires | Plus couteux qu'un modele lineaire |
| Ensemble aleatoire | `ExtraTreesClassifier` | Rapide, robuste, bon compromis performance / simplicite | Interpretabilite limitee aux importances de variables |
| Boosting | `XGBoost` | Puissant sur donnees tabulaires | Plus complexe, plus couteux, a retenir seulement si le gain est justifie |
| Reequilibrage | `SMOTE` + modeles classiques | Peut aider si classes desequilibrees | Cree des exemples artificiels, validation metier necessaire |
| Optimisation | `Optuna` + `RandomForestClassifier` | Automatise la recherche d'hyperparametres | Temps de calcul supplementaire, gain a valider hors optimisation |
| NLP pre-entraine | CamemBERT / analyse de sentiment | Exploite `retour_client` | Non retenu dans le modele principal car proche de la satisfaction et risque de fuite |

#### Type de resultat attendu

Le modele retenu produit deux types de sortie :

- une classe predite deterministe via `predict` : `insatisfait`, `neutre` ou `satisfait` ;
- des probabilites par classe via `predict_proba` lorsque le modele choisi le permet, utiles pour analyser le niveau de confiance du modele.

Dans ce projet, la sortie doit rester une aide a l'analyse qualite. Elle ne doit pas declencher automatiquement une decision commerciale individuelle.

#### Critere de decision

Le critere principal est le `macro_f1`, car il mesure mieux la performance globale lorsque les classes ne sont pas parfaitement equilibrees. L'`accuracy` et la `balanced_accuracy` restent suivies en complement.

Apres le nettoyage strict, le meilleur modele simple est celui qui arrive en tete de `results_post`. Les tests complementaires montrent que SMOTE peut ameliorer le post-voyage, mais ce choix doit etre confirme par validation croisee et validation metier avant industrialisation.

#### Contraintes operationnelles prises en compte

| Contrainte | Prise en compte |
| --- | --- |
| Ressources machine limitees | `n_jobs=1`, modeles tabulaires raisonnables, NLP lourd non retenu dans le modele final |
| Volume de donnees faible | Validation croisee et comparaison a une baseline pour eviter une conclusion trop optimiste |
| Besoin de reproductibilite | `random_state=42`, notebook final centralise, MLflow pour tracer les runs |
| Besoin de tracabilite | Model Card, MLflow, matrices de confusion, rapports de classification |
| Besoin metier | Separation stricte entre objectif pre-voyage et post-voyage |
| Passage en production | Export pipeline et API a traiter dans l'etape d'industrialisation |

#### Eco-conception

Les contraintes d'eco-conception sont portees a connaissance dans ce notebook :

- eviter les modeles lourds lorsque le gain de performance est faible ;
- limiter les hyperparametres des modeles principaux ;
- utiliser `n_jobs=1` pour eviter une consommation excessive sur une machine limitee ;
- ne pas retenir le NLP lourd dans le modele principal si le signal cree une fuite de donnees ;
- comparer les ameliorations avancees au modele simple avant de les retenir ;
- documenter le temps d'inference et la complexite du modele retenu.

#### Conclusion

Le choix du modele reste fonde sur la comparaison empirique, pas sur un algorithme impose. Le modele simple retenu est celui qui maximise le `macro_f1` dans `results_post`. SMOTE post-voyage devient un candidat d'amelioration, mais il n'est pas automatiquement industrialise sans validation complementaire.


In [ ]:
import time

from sklearn.metrics import auc, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Analyse ROC multiclasses du modele post-voyage retenu.
# Le ROC complete accuracy / macro_f1 en evaluant la separation probabiliste des classes.
if hasattr(best_post_pipeline, "predict_proba"):
    y_score_post = best_post_pipeline.predict_proba(X_test_post)
    y_test_binarized = label_binarize(y_test_post, classes=labels_3_classes)

    roc_auc_macro_ovr = roc_auc_score(
        y_test_post,
        y_score_post,
        multi_class="ovr",
        average="macro",
    )
    roc_auc_weighted_ovr = roc_auc_score(
        y_test_post,
        y_score_post,
        multi_class="ovr",
        average="weighted",
    )

    roc_rows = [
        {
            "metrique": "roc_auc_ovr_macro",
            "valeur": roc_auc_macro_ovr,
            "interpretation": "Moyenne non ponderee des AUC par classe",
        },
        {
            "metrique": "roc_auc_ovr_weighted",
            "valeur": roc_auc_weighted_ovr,
            "interpretation": "Moyenne ponderee par le support des classes",
        },
    ]

    for class_index, class_label in enumerate(labels_3_classes):
        fpr, tpr, _ = roc_curve(y_test_binarized[:, class_index], y_score_post[:, class_index])
        roc_rows.append({
            "metrique": f"roc_auc_{labels_readable[class_index]}",
            "valeur": auc(fpr, tpr),
            "interpretation": "AUC one-vs-rest de la classe",
        })

    roc_auc_post = pd.DataFrame(roc_rows)
    display(roc_auc_post.round(4))

    plt.figure(figsize=(8, 6))
    for class_index, class_label in enumerate(labels_3_classes):
        fpr, tpr, _ = roc_curve(y_test_binarized[:, class_index], y_score_post[:, class_index])
        plt.plot(
            fpr,
            tpr,
            label=f"{labels_readable[class_index]} - AUC={auc(fpr, tpr):.3f}",
        )

    plt.plot([0, 1], [0, 1], "--", color="gray", label="hasard")
    plt.title(f"Courbes ROC multiclasses - {best_model_name}")
    plt.xlabel("Taux de faux positifs")
    plt.ylabel("Taux de vrais positifs")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Le modele retenu ne fournit pas predict_proba : ROC AUC non calculable.")

# Mesure simple du temps d'inference sur le jeu de test.
start_time = time.perf_counter()
_ = best_post_pipeline.predict(X_test_post)
inference_time_seconds = time.perf_counter() - start_time

performance_operationnelle_post = pd.DataFrame([
    {
        "indicateur": "nb_lignes_test",
        "valeur": len(X_test_post),
        "interpretation": "Volume utilise pour mesurer l'inference",
    },
    {
        "indicateur": "temps_inference_total_secondes",
        "valeur": inference_time_seconds,
        "interpretation": "Temps total de prediction sur le jeu de test",
    },
    {
        "indicateur": "temps_inference_moyen_ms_ligne",
        "valeur": inference_time_seconds / len(X_test_post) * 1000,
        "interpretation": "Temps moyen de prediction par ligne",
    },
    {
        "indicateur": "cv_fit_time_moyen_secondes",
        "valeur": cv_results_post["fit_time"].mean(),
        "interpretation": "Temps moyen d'entrainement observe en validation croisee",
    },
    {
        "indicateur": "cv_score_time_moyen_secondes",
        "valeur": cv_results_post["score_time"].mean(),
        "interpretation": "Temps moyen d'evaluation observe en validation croisee",
    },
])

display(performance_operationnelle_post.round(6))



### 11.14 Entrainement supervise du modele IA

#### Methode d'entrainement retenue

| Element | Choix dans le projet | Justification |
| --- | --- | --- |
| Type d'apprentissage | Apprentissage supervise | La cible `satisfaction_client` est connue dans l'historique des sejours |
| Probleme IA | Classification multiclasses | Le modele post-voyage predit 3 classes : insatisfait, neutre, satisfait |
| Donnees d'entree | `X_post` | Variables pre-voyage + variables operationnelles post-voyage autorisees |
| Donnee cible | `y_post` | `satisfaction_client` regroupee en 3 classes |
| Split | Train/test stratifie 80/20 | Conserve la distribution des classes dans train et test |
| Pipeline | Pretraitement + modele dans `Pipeline` sklearn | Evite la fuite de donnees entre train et test pour imputation, traitement IQR des outliers, encodage et standardisation |
| Comparaison | Plusieurs modeles entraines automatiquement dans une boucle | Permet de comparer objectivement les familles d'algorithmes |
| Validation | Test set + validation croisee | Verifie la performance sur un jeu non vu et la stabilite du modele |

#### Optimisations testees

| Optimisation testee | Objectif | Decision actuelle |
| --- | --- | --- |
| `class_weight="balanced"` | Limiter l'effet du desequilibre des classes | Utilise dans les modeles compatibles |
| RandomForest regularise | Reduire l'overfitting par une profondeur faible et des feuilles plus larges | Conserve comme experience anti-overfitting |
| SMOTE pre-voyage | Tester un reequilibrage artificiel sur la cible 3 classes | Non retenu : le macro_f1 baisse de 0.3687 a 0.3523 |
| SMOTE post-voyage | Tester un reequilibrage artificiel sur un probleme contenant plus de signal | Candidat d'amelioration, a confirmer avant industrialisation |
| Optuna anti-overfitting | Automatiser la recherche d'hyperparametres avec penalite sur l'ecart train/validation et la complexite | Teste comme controle methodologique |
| XGBoost optimise | Tester une famille boosting performante sur tabulaire | Teste, a retenir uniquement si le gain justifie la complexite |
| NLP pre-entraine | Exploiter `retour_client` | Non retenu dans le modele principal pour eviter une fuite directe du ressenti client |

#### Reentrainement et transfert de connaissances

| Entrainement | Application dans ce projet |
| --- | --- |
| Modele entraine | Oui : les modeles sont entraines avec `.fit()` sur `X_train_post`, `y_train_post` via un pipeline supervise |
| Modele reentraine | Oui pour les experimentations : chaque modele compare ou teste est reentraine sur le meme jeu train |
| Reentrainement futur | Prevu lorsque de nouvelles donnees ou de nouvelles regles de nettoyage seront validees |
| Transfert de connaissances | Non applicable au modele principal tabulaire : les modeles sklearn ne transferent pas de poids d'un modele a l'autre |
| Transfert via modele pre-entraine | Teste conceptuellement avec NLP / modele de sentiment, mais non retenu dans le modele principal en raison du risque de fuite de donnees |

#### Feature engineering mobilise pour l'entrainement

Le feature engineering est utilise pour transformer des valeurs brutes en signaux metier plus exploitables :

- `budget_par_jour` : budget ramene a la duree du sejour ;
- `part_vol_budget` : poids du vol dans le budget total ;
- `sejour_long` : indicateur de sejour long ;
- `meteo_risque` : indicateur de meteo potentiellement defavorable ;
- `hebergement_luxe` : indicateur derive du type d'hebergement ;
- `destination_luxe` et `distance_vol_categorie` : enrichissement simple de la destination ;
- `imprevu_present`, `imprevu_transport`, `imprevu_meteo`, `budget_non_respecte`, `gravite_imprevu` : variables post-voyage explicatives reservees au modele post-voyage.

Les variables derivees reduisent la dependance a certaines valeurs brutes et rendent le dataset plus pertinent pour le cas d'usage metier. Les variables non retenues ou trop artificielles sont supprimees avant la modelisation.

#### Conclusion

Les modeles sont entraines automatiquement et superviseement, compares avec des indicateurs adaptes, optimises de maniere proportionnee au contexte, puis documentes avec leurs hyperparametres. Le reentrainement continu et l'export industriel du pipeline restent des etapes futures d'industrialisation.


In [ ]:
def extraire_hyperparametres_modele(model) -> dict:
    params = model.get_params()
    hyperparametres_utiles = [
        "strategy",
        "max_iter",
        "class_weight",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "min_samples_split",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "random_state",
        "n_jobs",
    ]
    return {key: params.get(key) for key in hyperparametres_utiles if key in params}


resume_entrainement_post = pd.DataFrame([
    {
        "element": "type_apprentissage",
        "valeur": "supervise",
    },
    {
        "element": "probleme",
        "valeur": "classification multiclasses 3 classes",
    },
    {
        "element": "taille_train",
        "valeur": len(X_train_post),
    },
    {
        "element": "taille_test",
        "valeur": len(X_test_post),
    },
    {
        "element": "nb_features_post",
        "valeur": len(feature_columns_post),
    },
    {
        "element": "modele_retenu",
        "valeur": best_model_name,
    },
])

display(resume_entrainement_post)

hyperparametres_modeles_post = pd.DataFrame([
    {
        "modele": model_name,
        "hyperparametres": extraire_hyperparametres_modele(model),
    }
    for model_name, model in modeles_post.items()
])

display(hyperparametres_modeles_post)

if "comparison_post" in globals():
    print("Comparaison incluant les tests complementaires :")
    display(comparison_post.round(4))
else:
    print("Comparaison des modeles principaux :")
    display(results_post.round(4))


### 11.15 Implementation et integration technique de la solution IA

Cette section formalise l'integration des briques technologiques dans l'environnement technique choisi afin de permettre l'exploitation progressive de la solution.

Le projet est actuellement dans une phase de notebook finalise et de socle technique minimal. L'industrialisation complete du modele de prediction a ete volontairement reportee tant que le notebook final n'est pas fige. En revanche, les briques de versioning, conteneurisation, API minimale, tests, tracking MLflow et CI/CD sont presentes ou preparees. ///...à revoir

#### Environnement technique retenu

| Brique | Outil / fichier | Role dans le projet | Statut |
| --- | --- | --- | --- |
| Langage | Python 3.12 | Analyse, preparation, entrainement, API | En place |
| Notebook | `notebooks/00_notebook_final_pre_post_voyage.ipynb` | Livrable central, experimentation, documentation | En place |
| API | `app/main.py` avec FastAPI | Socle applicatif minimal et point de sante `/health` | En place |
| Conteneurisation | `Dockerfile`, `docker-compose.yml` | Execution reproductible de l'API sur le port local `8001` | En place |
| Tracking | MLflow local avec `mlflow.db` | Tracabilite des runs, parametres, metriques et modele | En place localement |
| Tests | `tests/test_health.py` | Verification automatique du socle API | En place |
| CI/CD | `.github/workflows/ci-cd.yml` | Validation continue et build Docker a chaque push ou pull request | En place |
| Versioning | Git + GitHub | Suivi des versions du code, notebooks, docs et dataset synthetique | En place |

#### Processus de livraison et de deploiement continu

Le workflow GitHub Actions `.github/workflows/ci-cd.yml` met en place une chaine CI/CD minimale adaptee a l'etat actuel du projet.

| Etape du workflow | Objectif | Resultat attendu |
| --- | --- | --- |
| Checkout du depot | Recuperer la version Git a valider | Code source disponible dans l'environnement CI |
| Installation Python | Creer un environnement Python 3.12 | Environnement reproductible |
| Installation des dependances | Installer `requirements-dev.txt` | Dependances d'analyse, test et build disponibles |
| Validation Python | Executer `compileall` sur `app` et `tests` | Detecter les erreurs de syntaxe Python |
| Tests API | Executer `pytest -q` | Verifier que `/health` fonctionne |
| Validation notebook | Lire le notebook final et parser les cellules code avec `ast` | Detecter les erreurs de structure ou de syntaxe dans le livrable |
| Build Docker | Construire l'image `examen-ia:${github.sha}` | Verifier que le projet est packagable dans un conteneur |

Cette chaine correspond a une livraison continue minimale : chaque modification poussee sur GitHub est automatiquement verifiee et packagable. Le deploiement automatique vers un serveur distant n'est pas encore active, car le pipeline modele et l'endpoint `/predict` ne sont pas encore figes. //....

#### Versioning implemente

| Element versionne | Mode de versioning | Commentaire |
| --- | --- | --- |
| Code API | Git / GitHub | `app/main.py` est suivi dans le depot |
| Notebook final | Git / GitHub | Le livrable central est versionne |
| Documentation | Git / GitHub | Les fichiers `docs/` et `README.md` sont suivis |
| Dataset synthetique | Git / GitHub | Le fichier CSV est versionne car il est synthetique et anonymise |
| Experiences ML | MLflow local | Les runs, metriques et parametres sont traces localement |
| Artefacts locaux lourds | `.gitignore` | `mlflow.db`, `mlruns/`, `mlartifacts/`, `logs/`, `models/` ne sont pas pousses |
| Images Docker | Tag par SHA dans le workflow | Le build utilise `examen-ia:${github.sha}` pour relier l'image a une version Git |

#### Besoins d'integration documentes  /// à compléter

| Besoin d'integration | Decision actuelle | Evolution prevue |
| --- | --- | --- |
| Acces API | Endpoint `/health` disponible | Ajouter `/predict` lorsque le pipeline modele sera fige |
| Format d'entree prediction | Non active pour le moment | Definir un schema Pydantic base sur les features retenues |
| Format de sortie prediction | Non active pour le moment | Retourner classe predite, probabilites et version du modele |
| Suivi des predictions | Non active en production | Logger les predictions et surveiller les derives lorsque l'API modele sera restauree |
| Reporting modele | Notebook final + MLflow | Ajouter dashboard ou export de rapports si necessaire |
| Deploiement local | Docker Compose sur `localhost:8001` | Conserver pour tests locaux et demonstrations |
| Deploiement distant | Non active | A definir apres validation metier, RGPD et choix du pipeline final |

#### Limites actuelles 

- L'API actuelle est volontairement minimale : elle expose uniquement `/health`.
- Le modele n'est pas encore exporte comme artefact de production dans `models/`.
- L'endpoint `/predict` n'est pas encore restaure.
- Le suivi des predictions en production n'est pas encore actif.
- La CI/CD verifie et construit le projet, mais ne deploie pas encore vers un environnement distant.

Ces limites sont coherentes avec la decision projet : finaliser d'abord le notebook, les choix de modele et la documentation, puis restaurer l'industrialisation.

#### Conclusion

 Le projet dispose d'un versioning Git, d'une API minimale, d'une conteneurisation Docker, d'un tracking MLflow, de tests automatises et d'un workflow CI/CD. L'exploitation complete du modele avec endpoint `/predict`, monitoring des predictions et deploiement distant constitue l'etape suivante d'industrialisation.
//.....



## 12. Model Card - fiche du modele retenu

Cette fiche documente le modele post-voyage retenu pour la suite du projet. Elle doit etre mise a jour a chaque changement de donnees, de variables, d'algorithme ou d'hyperparametres.

### Details du modele

| Element | Description |
| --- | --- |
| Objectif | Predire / expliquer la satisfaction client apres sejour en 3 classes |
| Cible | `satisfaction_client` regroupee en `0 = insatisfait`, `1 = neutre`, `2 = satisfait` |
| Selection du modele | Meilleur modele selon `macro_f1` dans `results_post`, apres nettoyage strict |
| Pretraitement | Imputation, remplacement IQR des outliers numeriques continus, encodage OneHot des variables categorielles et standardisation via `ColumnTransformer` |
| Variables utilisees | Variables pre-voyage + variables post-voyage explicatives (`imprevus`, `reorganisation_necessaire`, `respect_budget`) |
| Variables exclues | `trip_id`, `satisfaction_client`, `retour_client` brut |
| Split d'evaluation | Train/test stratifie, 80 % entrainement et 20 % test |
| Validation complementaire | Validation croisee a 3 plis, matrice de confusion, rapport de classification, ROC multiclasses si `predict_proba` disponible |

### Architecture et hyperparametres

L'architecture retenue est un pipeline `scikit-learn` :

1. `ColumnTransformer` pour separer variables numeriques et categorielles ;
2. imputation, remplacement IQR des outliers continus et standardisation des numeriques ;
3. imputation et encodage OneHot des categorielles ;
4. classifieur supervise selectionne par comparaison empirique.

Les hyperparametres exacts du modele retenu sont extraits automatiquement dans la section d'entrainement via `extraire_hyperparametres_modele(best_post_pipeline)`.

### Performances globales

Les performances globales ne sont pas figees dans cette fiche, car elles changent apres chaque modification du nettoyage ou du feature engineering. Elles sont affichees dans :

- `results_post` pour les modeles simples ;
- `comparison_post` pour les tests complementaires ;
- `cv_summary_post` pour la validation croisee ;
- `overfitting_diagnostic` pour l'ecart train/test ;
- `report_post` et `confusion_post` pour le detail par classe.

Apres le nettoyage strict et le deplacement des pretraitements dans le pipeline, le meilleur modele simple obtient un `macro_f1` test autour de `0.44`. Optuna obtient environ `0.3983` en test et `0.3854` en validation croisee sur RandomForest, donc il n'est pas retenu. Les tests complementaires peuvent monter plus haut sur le split test, mais la validation croisee SMOTE reste moderee autour de `0.41`. Ces options doivent donc etre confirmees avant industrialisation.

### Performances par sous-groupes

La cellule suivante calcule les performances par sous-groupes metier. Les resultats doivent etre interpretes avec prudence lorsque le `support` est faible.

### Considerations ethiques et biais identifies

- Le dataset est synthetique : il ne garantit pas une representativite reelle des clients, destinations ou profils de voyageurs.
- Les variables comme `budget_total`, `destination`, `client_type` ou `type_hebergement` peuvent introduire des biais socio-economiques dans les recommandations.
- Le modele post-voyage utilise des evenements observes pendant ou apres le sejour ; il ne doit pas etre presente comme un outil de prediction avant depart.
- Les decisions commerciales ne doivent pas etre automatisees uniquement a partir du score du modele. Le resultat doit rester une aide a l'analyse.
- En cas d'utilisation de donnees reelles, les exigences RGPD, l'anonymisation et le controle des acces deviennent obligatoires.

### Cas d'usage recommandes

- Identifier les facteurs associes a une satisfaction faible apres sejour.
- Prioriser les dossiers necessitant une analyse qualite ou une action corrective.
- Comparer l'impact des imprevus, du respect du budget et des reorganisations sur l'experience client.
- Alimenter une demarche d'amelioration continue des offres de voyages.

### Limitations

- Les performances restent moderees : le modele est exploitable pour l'analyse, mais insuffisant pour une decision automatique individuelle.
- La prediction de la classe `neutre` reste fragile.
- Le modele depend fortement de variables post-voyage ; il ne repond donc pas au besoin pre-voyage de personnalisation avant depart.
- Les options d'augmentation de donnees comme SMOTE doivent etre validees metier, car elles creent des exemples artificiels.
- Les resultats doivent etre reevalues apres tout enrichissement du dataset, ajout de donnees reelles ou changement de cible metier.


In [ ]:
def evaluer_performance_sous_groupes(X_test, y_true, y_pred, colonnes_sous_groupes, min_support=20):
    donnees_eval = X_test.reset_index(drop=True).copy()
    y_true_eval = pd.Series(y_true).reset_index(drop=True)
    y_pred_eval = pd.Series(y_pred).reset_index(drop=True)

    lignes = []
    for colonne in colonnes_sous_groupes:
        if colonne not in donnees_eval.columns:
            continue

        for valeur, index_groupe in donnees_eval.groupby(colonne, dropna=False).groups.items():
            index_groupe = list(index_groupe)
            support = len(index_groupe)
            if support < min_support:
                continue

            y_true_groupe = y_true_eval.loc[index_groupe]
            y_pred_groupe = y_pred_eval.loc[index_groupe]

            lignes.append({
                "sous_groupe": colonne,
                "valeur": valeur,
                "support": support,
                "accuracy": accuracy_score(y_true_groupe, y_pred_groupe),
                "balanced_accuracy": balanced_accuracy_score(y_true_groupe, y_pred_groupe),
                "macro_f1": f1_score(
                    y_true_groupe,
                    y_pred_groupe,
                    labels=labels_3_classes,
                    average="macro",
                    zero_division=0,
                ),
            })

    return (
        pd.DataFrame(lignes)
        .sort_values(["sous_groupe", "macro_f1"], ascending=[True, False])
        .reset_index(drop=True)
    )


colonnes_sous_groupes_model_card = [
    "client_type",
    "respect_budget",
    "imprevus",
    "reorganisation_necessaire",
    "meteo_prevue",
]

performances_sous_groupes_post = evaluer_performance_sous_groupes(
    X_test_post,
    y_test_post,
    best_predictions,
    colonnes_sous_groupes_model_card,
    min_support=20,
)

display(performances_sous_groupes_post.round(4))



## 13. Synth?se finale

| Axe | Conclusion |
| --- | --- |
| Pr?-voyage | Mod?le d?sormais ?valu? en 3 classes ; meilleur mod?le `ExtraTrees_pre`, `macro_f1 = 0.3687`, performance encore limit?e par le faible signal disponible avant d?part. |
| Post-voyage | Mod?le plus pertinent en 3 classes ; meilleur mod?le simple `LogisticRegression_3_classes`, `macro_f1 = 0.4392`, car les variables op?rationnelles expliquent mieux la satisfaction. |
| Nettoyage strict | Les incoh?rences `prix_vol > budget_total`, cible invalide et contradictions `imprevus` / `reorganisation_necessaire` sont supprim?es avant les deux mod?lisations. |
| Pipeline sans fuite | Imputation, traitement IQR des outliers, standardisation, encodage et SMOTE ?ventuel sont r?alis?s apr?s split dans les pipelines. |
| NLP `retour_client` | Utile pour l'analyse qualitative, mais non retenu dans le mod?le principal car trop proche de la satisfaction. |
| SMOTE | Non utile en pr?-voyage ; candidat d'am?lioration en post-voyage, ? confirmer avant industrialisation. |

Le mod?le ? pr?parer pour l'?tape suivante est donc le meilleur mod?le post-voyage 3 classes sans texte libre, apr?s validation des choix de r??quilibrage et de complexit?.


## 14. Références documentaires du projet


In [ ]:
documents = pd.DataFrame([
    {
        "document": "docs/etat_projet.md",
        "role": "historique technique du projet, environnement, Docker, Git, Jupyter",
    },
    {
        "document": "docs/objectif_1_dataset.md",
        "role": "identification du dataset, besoins métiers, cas d'usage, datasheet",
    },
    {
        "document": "docs/experiences_modelisation.md",
        "role": "détail des expériences de modélisation et conclusions",
    },
    {
        "document": "notebooks/Exam.ipynb",
        "role": "notebook pré-voyage détaillé et historique",
    },
    {
        "document": "notebooks/objectif_2_post_voyage.ipynb",
        "role": "notebook post-voyage détaillé et historique",
    },
])

display(documents)